# 02 — Population municipale 2035-2050

Sources : `exctraction of data/output/CSV_final.csv` (recensements 2001/2012/2024),
`data_ine/INE_Bolivia_Proyecciones_Poblacion_Departamento_Provincia_Municipio_TIOC_2024_2034.csv`
(projections municipales 2024-2034), `data_ine/INE_Bolivia_Poblacion_Estimada_Proyectada_y_Tasas_Crecimiento_2000_2050.csv`
(population et taux nationaux 2000-2050).

Même clé que `01_base_municipale.ipynb` : `DEPARTAMENTO` + `PROVINCIA` + `MUNICIPIO/TIOC`
(nécessaire pour distinguer les deux "Santa Rosa" du périmètre : Beni/General José Ballivián
et Pando/Abuná).

In [1]:
import os
import io
import math

import numpy as np
import pandas as pd
from scipy.optimize import brentq

PROJ = os.getcwd()
REPO = os.path.dirname(PROJ)
CSV_FINAL = os.path.join(REPO, "exctraction of data", "output", "CSV_final.csv")
INE_MUNI_PATH = os.path.join(PROJ, "data_ine", "INE_Bolivia_Proyecciones_Poblacion_Departamento_Provincia_Municipio_TIOC_2024_2034.csv")
INE_NAT_PATH = os.path.join(PROJ, "data_ine", "INE_Bolivia_Poblacion_Estimada_Proyectada_y_Tasas_Crecimiento_2000_2050.csv")

KEYS = ["DEPARTAMENTO", "PROVINCIA", "MUNICIPIO/TIOC"]
POP_BLOCK = "NÚMERO DE PERSONAS POR FUENTE DE ELECTRICIDAD"
EXPECTED_N_MUNICIPIOS = 21

## 1. Population municipale 2001/2012/2024 (CSV_final)

In [2]:
raw = pd.read_csv(CSV_FINAL, encoding="utf-8")
municipios = raw[raw["MUNICIPIO/TIOC"].notna() & (raw["MUNICIPIO/TIOC"].astype(str).str.strip() != "")].copy()
assert len(municipios) == EXPECTED_N_MUNICIPIOS, len(municipios)

base = municipios[KEYS].copy()
for year in (2001, 2012, 2024):
    col = f"{POP_BLOCK} | {year} | Total"
    base[f"pop_{year}_censo"] = municipios[col].astype(float)
base = base.reset_index(drop=True)

amazonia = raw[raw["DEPARTAMENTO"] == "Amazonía Norte"]
pop_region = {year: float(amazonia[f"{POP_BLOCK} | {year} | Total"].iloc[0]) for year in (2001, 2012, 2024)}
base

,DEPARTAMENTO,PROVINCIA,MUNICIPIO/TIOC,pop_2001_censo,pop_2012_censo,pop_2024_censo
0,La Paz,Abel Iturralde,Ixiamas,5207.0,8338.0,11330.0
1,Beni,Vaca Diez,Riberalta,73981.0,86903.0,107816.0
2,Beni,Vaca Diez,Guayaramerín,39173.0,40186.0,40130.0
3,Beni,General José Ballivián,Reyes,11016.0,12912.0,11284.0
4,Beni,General José Ballivián,Santa Rosa,8915.0,9297.0,10953.0
5,Beni,Yacuma,Exaltación,6504.0,5996.0,7810.0
6,Pando,Nicolás Suárez,Cobija,20873.0,43616.0,51908.0
7,Pando,Nicolás Suárez,Porvenir,3554.0,7653.0,9096.0
8,Pando,Nicolás Suárez,Bolpebra,1137.0,2058.0,2338.0
9,Pando,Nicolás Suárez,Bella Flor,2140.0,3550.0,3421.0


## 2. INE municipal 2024-2034

Le fichier source contient un octet UTF-8 corrompu dans l'en-tête `ÁREA` (non utilisée) et
4 lignes de pied de page ("Fuente : …", "RECOMENDACIÓN : …") avec des virgules dans le texte
qui cassent le parsing en colonnes fixes. Décodage en `errors="replace"`, puis chaque ligne mal
tokenisée est interceptée et affichée explicitement (rien n'est ignoré silencieusement).

In [3]:
text = open(INE_MUNI_PATH, "rb").read().decode("utf-8", errors="replace")

bad_lines = []
def report_bad_line(line):
    bad_lines.append(line)
    return None

ine_muni_raw = pd.read_csv(io.StringIO(text), sep=",", engine="python", on_bad_lines=report_bad_line)
ine_muni_raw.columns = ine_muni_raw.columns.str.strip()

print(f"{len(bad_lines)} ligne(s) non tokenisée(s), écartée(s) :")
for l in bad_lines:
    print(" ", l)

3 ligne(s) non tokenisée(s), écartée(s) :
  ['Fuente: Ministerio de Educación', ' Ministerio de Salud y Deportes', ' Instituto Nacional de Estadística. Estimaciones y proyecciones de población', ' Revisión 2025', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']
  ['RECOMENDACIÓN: Las proyecciones de población son elaboradas con base a información sobre los componentes demográficos (fecundidad', ' mortalidad y migración) investigadas en los censos y encuestas de demografía y salud. ', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']
  ['Cada Revisión de Proyección incorpora\xa0 en el momento de su realización información más reciente sobre los componentes demográficos y/o cambios metodológicos de cálculo de proyecciones', ' debidamente explicitados en respectivas ', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']


In [4]:
ine_muni = ine_muni_raw[ine_muni_raw["NIVEL"] == "Municipio/TIOC"].copy()
assert len(ine_muni) == 342, len(ine_muni)

YEARS_HIST = list(range(2024, 2035))
m = base.merge(ine_muni[KEYS + [str(y) for y in YEARS_HIST]], on=KEYS, how="left")

missing = m[m["2024"].isna()]
assert len(missing) == 0, f"Municipalité(s) absente(s) de l'INE municipal : {missing[KEYS].to_dict('records')}"
assert len(m) == EXPECTED_N_MUNICIPIOS
m

,DEPARTAMENTO,PROVINCIA,MUNICIPIO/TIOC,pop_2001_censo,pop_2012_censo,pop_2024_censo,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034
0,La Paz,Abel Iturralde,Ixiamas,5207.0,8338.0,11330.0,12933.0,13109.0,13264.0,13406.0,13538.0,13645.0,13741.0,13824.0,13889.0,13943.0,13990.0
1,Beni,Vaca Diez,Riberalta,73981.0,86903.0,107816.0,121074.0,122321.0,123652.0,124905.0,126080.0,127268.0,128461.0,129694.0,130875.0,132080.0,133278.0
2,Beni,Vaca Diez,Guayaramerín,39173.0,40186.0,40130.0,46337.0,46247.0,46277.0,46366.0,46498.0,46677.0,46914.0,47228.0,47556.0,47923.0,48320.0
3,Beni,General José Ballivián,Reyes,11016.0,12912.0,11284.0,12695.0,12539.0,12415.0,12312.0,12238.0,12176.0,12124.0,12113.0,12102.0,12100.0,12115.0
4,Beni,General José Ballivián,Santa Rosa,8915.0,9297.0,10953.0,12221.0,12308.0,12407.0,12510.0,12614.0,12709.0,12819.0,12933.0,13041.0,13144.0,13270.0
5,Beni,Yacuma,Exaltación,6504.0,5996.0,7810.0,8799.0,8894.0,8974.0,9067.0,9158.0,9243.0,9333.0,9414.0,9497.0,9591.0,9676.0
6,Pando,Nicolás Suárez,Cobija,20873.0,43616.0,51908.0,59682.0,60586.0,61363.0,62179.0,62912.0,63632.0,64301.0,64944.0,65553.0,66190.0,66799.0
7,Pando,Nicolás Suárez,Porvenir,3554.0,7653.0,9096.0,10121.0,10271.0,10413.0,10548.0,10671.0,10787.0,10887.0,10993.0,11096.0,11199.0,11302.0
8,Pando,Nicolás Suárez,Bolpebra,1137.0,2058.0,2338.0,2640.0,2663.0,2700.0,2721.0,2751.0,2770.0,2797.0,2815.0,2839.0,2859.0,2891.0
9,Pando,Nicolás Suárez,Bella Flor,2140.0,3550.0,3421.0,3948.0,3938.0,3925.0,3916.0,3904.0,3889.0,3889.0,3870.0,3864.0,3872.0,3866.0


## 3. Indice municipal I(t), 2024-2034

`I(t) = P_INE(t) / P_INE(2024)`. Note : `P_INE(2024)` (projection INE) diffère du Censo 2024 de
`CSV_final` (ex. Riberalta : 121 074 projeté vs 107 816 recensé) — I(t) n'est utilisé que comme
indice relatif, jamais comme niveau absolu, d'où le rebasage sur `pop_2024_censo` à l'étape 6.

In [5]:
for y in YEARS_HIST:
    m[f"I_{y}"] = m[str(y)] / m["2024"]

assert (m["I_2024"] == 1.0).all()
m[KEYS + [f"I_{y}" for y in YEARS_HIST]]

,DEPARTAMENTO,PROVINCIA,MUNICIPIO/TIOC,I_2024,I_2025,I_2026,I_2027,I_2028,I_2029,I_2030,I_2031,I_2032,I_2033,I_2034
0,La Paz,Abel Iturralde,Ixiamas,1.0,1.013609,1.025593,1.036573,1.046780,1.055053,1.062476,1.068894,1.073919,1.078095,1.081729
1,Beni,Vaca Diez,Riberalta,1.0,1.010299,1.021293,1.031642,1.041347,1.051159,1.061012,1.071196,1.080950,1.090903,1.100798
2,Beni,Vaca Diez,Guayaramerín,1.0,0.998058,0.998705,1.000626,1.003475,1.007338,1.012452,1.019229,1.026307,1.034228,1.042795
3,Beni,General José Ballivián,Reyes,1.0,0.987712,0.977944,0.969831,0.964002,0.959118,0.955022,0.954155,0.953289,0.953131,0.954313
4,Beni,General José Ballivián,Santa Rosa,1.0,1.007119,1.015220,1.023648,1.032158,1.039931,1.048932,1.058260,1.067098,1.075526,1.085836
5,Beni,Yacuma,Exaltación,1.0,1.010797,1.019889,1.030458,1.040800,1.050460,1.060689,1.069894,1.079327,1.090010,1.099670
6,Pando,Nicolás Suárez,Cobija,1.0,1.015147,1.028166,1.041838,1.054120,1.066184,1.077394,1.088167,1.098371,1.109045,1.119249
7,Pando,Nicolás Suárez,Porvenir,1.0,1.014821,1.028851,1.042190,1.054342,1.065804,1.075684,1.086157,1.096334,1.106511,1.116688
8,Pando,Nicolás Suárez,Bolpebra,1.0,1.008712,1.022727,1.030682,1.042045,1.049242,1.059470,1.066288,1.075379,1.082955,1.095076
9,Pando,Nicolás Suárez,Bella Flor,1.0,0.997467,0.994174,0.991895,0.988855,0.985056,0.985056,0.980243,0.978723,0.980750,0.979230


## 4. Taux national et profil de décélération rho(t), lissé 2032-2034

In [6]:
nat = pd.read_csv(INE_NAT_PATH, encoding="utf-8")
nat = nat[pd.to_numeric(nat["AÑO"], errors="coerce").notna()].copy()
nat["AÑO"] = nat["AÑO"].astype(int)

RATE_COL = "TASA ANUAL DE CRECIMIENTO DE POBLACIÓN (%) | Anual"
rate_nat = nat.set_index("AÑO")[RATE_COL]

ref = rate_nat.loc[[2032, 2033, 2034]].mean()
YEARS_FUT = list(range(2035, 2051))
rho = {y: rate_nat.loc[y] / ref for y in YEARS_FUT}

assert all(v > 0 for v in rho.values()), "rho negatif detecte"

for y in (2035, 2040, 2045, 2050):
    print(f"rho({y}) = {rho[y]:.4f}")

rho(2035) = 0.9719
rho(2040) = 0.9545
rho(2045) = 0.6600
rho(2050) = 0.0921


## 5-6. Prolongement 2035-2050 et population finale

`taux(2034) = 100·ln(P(2034)/P(2033))` par municipalité, puis `taux(t) = taux(2034)·rho(t)` et
`P(t) = P(t-1)·(1+taux(t)/100)` pour t=2035..2050. Pour t≤2034 : `P(t) = pop_2024_censo · I(t)`.

In [7]:
pop = pd.DataFrame(index=m.index, columns=range(2024, 2051), dtype=float)

for y in YEARS_HIST:
    pop[y] = m["pop_2024_censo"] * m[f"I_{y}"]

taux_2034 = 100 * np.log(m["2034"] / m["2033"])
prev = pop[2034].copy()
for y in YEARS_FUT:
    taux_y = taux_2034 * rho[y]
    prev = prev * (1 + taux_y / 100)
    pop[y] = prev

result = pd.concat([m[KEYS], pop[[2035, 2050]].rename(columns={2035: "pop_2035", 2050: "pop_2050"})], axis=1)
result

,DEPARTAMENTO,PROVINCIA,MUNICIPIO/TIOC,pop_2035,pop_2050
0,La Paz,Abel Iturralde,Ixiamas,12296.072825,12741.251338
1,Beni,Vaca Diez,Riberalta,119725.131655,131683.701488
2,Beni,Vaca Diez,Guayaramerín,42182.904652,46017.859874
3,Beni,General José Ballivián,Reyes,10781.430736,10923.648726
4,Beni,General José Ballivián,Santa Rosa,12003.436352,13273.440312
5,Beni,Yacuma,Exaltación,8662.074839,9506.683396
6,Pando,Nicolás Suárez,Cobija,58615.103461,64557.438773
7,Pando,Nicolás Suárez,Porvenir,10247.773343,11286.266084
8,Pando,Nicolás Suárez,Bolpebra,2587.983323,2909.951957
9,Pando,Nicolás Suárez,Bella Flor,3344.896796,3290.412970


## Assertions bloquantes

In [8]:
# a. 21 municipalités trouvées dans les deux sources
assert len(m) == EXPECTED_N_MUNICIPIOS

# b. I(2024) == 1.000 pour toutes
assert (m["I_2024"] == 1.0).all()

# c. Aucune population 2050 negative ou nulle
assert (pop[2050] > 0).all()

# d. Somme régionale croissante et monotone 2024 -> 2050
region_sum = pop.sum(axis=0)
assert region_sum.diff().dropna().gt(0).all()

print("Assertions OK.")
region_sum[[2024, 2035, 2050]]

Assertions OK.


2024    317517.000000
2035    349535.185800
2050    381276.663493
dtype: float64

## Étape 2 — validation rétrospective, 5 modèles (protocole INE §7.2 point 5)

Sur `CSV_final` uniquement (pas de fichiers INE 2024-2034/2000-2050 ici). Calibration sur les
recensements 2001 et 2012, prédiction 2024, comparaison au Censo 2024 observé.

- **Linéaire** : `P(2024) = P(2012) + (P(2012)-P(2001))/11 · 12`
- **Exponentiel** : `P(2024) = P(2001) · (P(2012)/P(2001))^(23/11)`
- **Share of Growth** : `part_i = (P_i(2012)-P_i(2001)) / (P_région(2012)-P_région(2001))`,
  `P_i(2024) = P_i(2012) + part_i · (P_région(2024)-P_région(2012))`, contrôle = Amazonía Norte.
- **Logistique** : `P(t) = K/(1+a·e^(-b(t-2001)))`, auto-inflexion `K = 2·P(2012)`, `a,b` résolus
  à partir de `P(2001)` et `P(2012)`.
- **Logistique proportionnel** : `K_région` résolu exactement sur les 3 recensements régionaux
  (2001/2012/2024, Amazonía Norte) ; `K_i = K_région · P_i(2012)/P_région(2012)` ; `a_i,b_i`
  résolus à partir de `P_i(2001)`, `P_i(2012)` et `K_i` fixé.

In [9]:
def solve_ab(K, P_t0, P_t1, h1):
    a = K / P_t0 - 1
    inner = (K / P_t1 - 1) / a
    if inner <= 0 or inner >= 1:
        return None
    b = -math.log(inner) / h1
    return a, b

def logistic(K, a, b, t, t0):
    return K / (1 + a * math.exp(-b * (t - t0)))

P0r, P1r, P2r = pop_region[2001], pop_region[2012], pop_region[2024]

def region_resid(K):
    a, b = solve_ab(K, P0r, P1r, 11)
    return logistic(K, a, b, 2024, 2001) - P2r

K_region = brentq(region_resid, P2r * 1.0001, P2r * 50)
a_region, b_region = solve_ab(K_region, P0r, P1r, 11)
K_region

354750.89810173726

In [10]:
domain_fail = []
rows = []
for _, row in base.iterrows():
    P0, P1, P24 = row["pop_2001_censo"], row["pop_2012_censo"], row["pop_2024_censo"]
    name = row["MUNICIPIO/TIOC"]

    pred_lin = P1 + (P1 - P0) / 11 * 12
    pred_exp = P0 * (P1 / P0) ** (23 / 11)

    part = (P1 - P0) / (P1r - P0r)
    pred_sog = P1 + part * (P2r - P1r)

    ab = solve_ab(2 * P1, P0, P1, 11)
    if ab is None:
        pred_log = np.nan
        domain_fail.append((name, "logistique"))
    else:
        pred_log = logistic(2 * P1, ab[0], ab[1], 2024, 2001)

    Ki = K_region * (P1 / P1r)
    ab2 = solve_ab(Ki, P0, P1, 11)
    if ab2 is None:
        pred_logp = np.nan
        domain_fail.append((name, "logistique_proportionnel"))
    else:
        pred_logp = logistic(Ki, ab2[0], ab2[1], 2024, 2001)

    rows.append(dict(municipio=name, P2024_obs=P24, lineaire=pred_lin, exponentiel=pred_exp,
                      share_of_growth=pred_sog, logistique=pred_log, logistique_proportionnel=pred_logp))

print(f"{len(domain_fail)} échec(s) de domaine (courbe logistique non résoluble avec K assigné) :")
for name, model in domain_fail:
    print(" ", name, "-", model)

retro = pd.DataFrame(rows)
MODELS = ["lineaire", "exponentiel", "share_of_growth", "logistique", "logistique_proportionnel"]
retro

2 échec(s) de domaine (courbe logistique non résoluble avec K assigné) :
  Exaltación - logistique
  Exaltación - logistique_proportionnel


,municipio,P2024_obs,lineaire,exponentiel,share_of_growth,logistique,logistique_proportionnel
0,Ixiamas,11330.0,11753.636364,13935.571305,10455.625049,11722.548752,10189.929673
1,Riberalta,107816.0,100999.727273,103586.989362,95642.684090,100979.924298,97516.470191
2,Guayaramerín,40130.0,41291.090909,41320.990413,40871.133879,41291.046413,41233.545064
3,Reyes,11284.0,14980.363636,15354.407776,14194.343371,14977.530231,14474.157298
4,Santa Rosa,10953.0,9713.727273,9732.419336,9555.362430,9713.682686,9679.422755
5,Exaltación,7810.0,5441.818182,5486.961411,5652.418548,NaN,NaN
6,Cobija,51908.0,68426.545455,97454.802615,58998.033375,67982.578687,54972.656626
7,Porvenir,9096.0,12124.636364,17669.697739,10425.323563,12040.010186,9669.027985
8,Bolpebra,2338.0,3062.727273,3931.480960,2680.910466,3049.628165,2557.484106
9,Bella Flor,3421.0,5088.181818,6166.320252,4503.641431,5072.482328,4361.911548


In [11]:
for mo in MODELS:
    retro[f"APE_{mo}"] = 100 * (retro[mo] - retro["P2024_obs"]).abs() / retro["P2024_obs"]

ape_table = retro[["municipio"] + [f"APE_{mo}" for mo in MODELS]]
ape_table

,municipio,APE_lineaire,APE_exponentiel,APE_share_of_growth,APE_logistique,APE_logistique_proportionnel
0,Ixiamas,3.739068,22.997099,7.717343,3.464684,10.062404
1,Riberalta,6.322135,3.922433,11.290825,6.340502,9.552877
2,Guayaramerín,2.893324,2.967831,1.846832,2.893213,2.749925
3,Reyes,32.757565,36.072384,25.791770,32.732455,28.271511
4,Santa Rosa,11.314459,11.143802,12.760317,11.314866,11.627657
5,Exaltación,30.322430,29.744412,27.625883,NaN,NaN
6,Cobija,31.822735,87.745247,13.658845,30.967440,5.904016
7,Porvenir,33.296354,94.257891,14.614375,32.365987,6.299780
8,Bolpebra,30.997745,68.155730,14.666829,30.437475,9.387686
9,Bella Flor,48.733757,80.249057,31.646929,48.274841,27.503991


In [12]:
stats = pd.DataFrame({
    mo: {
        "mean_APE": retro[f"APE_{mo}"].mean(skipna=True),
        "median_APE": retro[f"APE_{mo}"].median(skipna=True),
        "CV_error": retro[f"APE_{mo}"].std(skipna=True) / retro[f"APE_{mo}"].mean(skipna=True),
    } for mo in MODELS
}).T

classement = stats.sort_values("mean_APE").reset_index().rename(columns={"index": "modele"})
classement.index = classement.index + 1
classement

,modele,mean_APE,median_APE,CV_error
1,logistique_proportionnel,15.416145,11.915060,0.806110
2,share_of_growth,19.956239,14.614375,0.840133
3,logistique,28.621508,22.351874,0.954559
4,lineaire,29.414116,30.322430,0.932811
5,exponentiel,88.717782,45.582358,1.070447


## Étape 3 — diagnostics complémentaires

### 3.1 Erreur pondérée par population 2024

Même `retro` qu'à l'étape 2. Erreur moyenne pondérée par `P2024_obs`, classement non pondéré et
pondéré côte à côte.

In [13]:
weights = retro["P2024_obs"]
weighted_mean = {}
for mo in MODELS:
    s = retro[f"APE_{mo}"]
    valid = s.notna()
    weighted_mean[mo] = (s[valid] * weights[valid]).sum() / weights[valid].sum()

unweighted_rank = retro[[f"APE_{mo}" for mo in MODELS]].mean(skipna=True).rename("unweighted_mean_APE")
unweighted_rank = unweighted_rank.sort_values().rename_axis("modele").reset_index()
unweighted_rank["modele"] = unweighted_rank["modele"].str.replace("APE_", "")

weighted_rank = pd.Series(weighted_mean, name="weighted_mean_APE").sort_values().rename_axis("modele").reset_index()

classement_side_by_side = pd.concat([unweighted_rank, weighted_rank], axis=1, keys=["non pondéré", "pondéré"])
classement_side_by_side.index = classement_side_by_side.index + 1
classement_side_by_side

non pondéré                                       pondéré  \
                     modele unweighted_mean_APE                    modele   
1  logistique_proportionnel           15.416145  logistique_proportionnel   
2           share_of_growth           19.956239           share_of_growth   
3                logistique           28.621508                logistique   
4                  lineaire           29.414116                  lineaire   
5               exponentiel           88.717782               exponentiel   

                     
  weighted_mean_APE  
1          9.629062  
2         12.425564  
3         15.695509  
4         16.395944  
5         43.876790

### 3.2 Erreur sur les agrégats

Mapping cluster lu depuis `Clustering/output/clustering_results.csv` (colonne `Municipio`, noms
séparés par `_`, suffixe département pour les deux "Santa Rosa").

In [14]:
CLUSTER_PATH = os.path.join(REPO, "Clustering", "output", "clustering_results.csv")
cluster_raw = pd.read_csv(CLUSTER_PATH)

def cluster_key(dept, muni):
    if muni == "Santa Rosa":
        return f"Santa_Rosa_{'Beni' if dept == 'Beni' else 'Pando'}"
    return muni.replace(" ", "_")

base_cl = base.copy()
base_cl["cluster_key"] = [cluster_key(d, mu) for d, mu in zip(base_cl["DEPARTAMENTO"], base_cl["MUNICIPIO/TIOC"])]
base_cl = base_cl.merge(cluster_raw[["Municipio", "Cluster"]], left_on="cluster_key", right_on="Municipio", how="left")
assert base_cl["Cluster"].isna().sum() == 0, base_cl[base_cl["Cluster"].isna()][KEYS]
base_cl["cluster_label"] = "C" + (base_cl["Cluster"] + 1).astype(int).astype(str)

retro["cluster_label"] = base_cl["cluster_label"].values
base_cl[["MUNICIPIO/TIOC", "cluster_label"]]

,MUNICIPIO/TIOC,cluster_label
0,Ixiamas,C1
1,Riberalta,C3
2,Guayaramerín,C3
3,Reyes,C1
4,Santa Rosa,C1
5,Exaltación,C1
6,Cobija,C5
7,Porvenir,C4
8,Bolpebra,C2
9,Bella Flor,C4


In [15]:
agg_total = []
for mo in MODELS:
    valid = retro[mo].notna()
    pred_sum = retro.loc[valid, mo].sum()
    n_exclu = (~valid).sum()
    ecart = 100 * (pred_sum - pop_region[2024]) / pop_region[2024]
    agg_total.append(dict(modele=mo, pred_sum=pred_sum, ecart_pct=ecart, n_exclu=n_exclu))

pd.DataFrame(agg_total)

,modele,pred_sum,ecart_pct,n_exclu
0,lineaire,348012.181818,9.604267e+00,0
1,exponentiel,440974.111045,3.888205e+01,0
2,share_of_growth,317517.000000,1.833214e-14,0
3,logistique,341280.019985,7.484015e+00,1
4,logistique_proportionnel,304100.752494,-4.225364e+00,1


In [16]:
cluster_obs = retro.groupby("cluster_label")["P2024_obs"].sum()

tab = {}
for mo in MODELS:
    valid = retro[mo].notna()
    pred_by_cluster = retro[valid].groupby("cluster_label")[mo].sum()
    tab[mo] = 100 * (pred_by_cluster - cluster_obs) / cluster_obs

print("Exaltación (C1) exclue des sommes logistique / logistique_proportionnel (échec de domaine, étape 2).")
pd.DataFrame(tab).reindex(sorted(cluster_obs.index))

Exaltación (C1) exclue des sommes logistique / logistique_proportionnel (échec de domaine, étape 2).


,lineaire,exponentiel,share_of_growth,logistique,logistique_proportionnel
cluster_label,,,,,
C1,1.238721,7.570292,-3.671727,-11.995162,-16.998551
C2,30.997745,68.155730,14.666829,30.437475,9.387686
C3,-3.546232,1.049354,-8.164711,-3.603908,-7.178619
C4,29.592380,114.987758,11.458545,28.455183,2.890890
C5,31.822735,87.745247,13.658845,30.967440,5.904016


### 3.3 D'où vient l'erreur — modèle logistique proportionnel

In [17]:
retro["taux_0112"] = 100 * np.log(base["pop_2012_censo"] / base["pop_2001_censo"]) / 11
sub = retro.dropna(subset=["APE_logistique_proportionnel"])

top5 = sub.nlargest(5, "APE_logistique_proportionnel")[["municipio", "APE_logistique_proportionnel", "P2024_obs", "taux_0112"]]
bot5 = sub.nsmallest(5, "APE_logistique_proportionnel")[["municipio", "APE_logistique_proportionnel", "P2024_obs", "taux_0112"]]

print(f"Constat : taux 2001-2012 moyen top5 = {top5['taux_0112'].mean():.2f} %/an, bottom5 = {bot5['taux_0112'].mean():.2f} %/an.")
top5

Constat : taux 2001-2012 moyen top5 = 6.08 %/an, bottom5 = 5.94 %/an.


,municipio,APE_logistique_proportionnel,P2024_obs,taux_0112
18,Nueva Esperanza,44.883298,1572.0,9.073151
19,Villa Nueva,44.144661,2485.0,9.613536
3,Reyes,28.271511,11284.0,1.443712
9,Bella Flor,27.503991,3421.0,4.601289
17,Ingavi,20.723889,2545.0,5.671883


In [18]:
bot5

,municipio,APE_logistique_proportionnel,P2024_obs,taux_0112
10,Puerto Rico,1.286663,6870.0,3.478846
2,Guayaramerín,2.749925,40130.0,0.232099
15,Sena,3.424059,10247.0,12.327289
6,Cobija,5.904016,51908.0,6.699705
7,Porvenir,6.299780,9096.0,6.972945


### 3.4 Contrôle sur la projection principale

Taux annuel implicite : `100·ln(P_fin/P_deb)/n_années` (11 ans 2024-2035, 15 ans 2035-2050).

In [19]:
ctrl = m[KEYS].copy()
ctrl["cluster_label"] = base_cl["cluster_label"].values
ctrl["pop_2024"] = pop[2024].values
ctrl["pop_2035"] = pop[2035].values
ctrl["pop_2050"] = pop[2050].values
ctrl["taux_2435"] = 100 * np.log(ctrl["pop_2035"] / ctrl["pop_2024"]) / 11
ctrl["taux_3550"] = 100 * np.log(ctrl["pop_2050"] / ctrl["pop_2035"]) / 15
ctrl["sign_change"] = (ctrl["taux_2435"] * ctrl["taux_3550"]) < 0
ctrl

,DEPARTAMENTO,PROVINCIA,MUNICIPIO/TIOC,cluster_label,pop_2024,pop_2035,pop_2050,taux_2435,taux_3550,sign_change
0,La Paz,Abel Iturralde,Ixiamas,C1,11330.0,12296.072825,12741.251338,0.743871,0.237100,False
1,Beni,Vaca Diez,Riberalta,C3,107816.0,119725.131655,131683.701488,0.952477,0.634695,False
2,Beni,Vaca Diez,Guayaramerín,C3,40130.0,42182.904652,46017.859874,0.453553,0.580097,False
3,Beni,General José Ballivián,Reyes,C1,11284.0,10781.430736,10923.648726,-0.414187,0.087365,True
4,Beni,General José Ballivián,Santa Rosa,C1,10953.0,12003.436352,13273.440312,0.832542,0.670481,False
5,Beni,Yacuma,Exaltación,C1,7810.0,8662.074839,9506.683396,0.941357,0.620272,False
6,Pando,Nicolás Suárez,Cobija,C5,51908.0,58615.103461,64557.438773,1.104723,0.643753,False
7,Pando,Nicolás Suárez,Porvenir,C4,9096.0,10247.773343,11286.266084,1.083870,0.643508,False
8,Pando,Nicolás Suárez,Bolpebra,C2,2338.0,2587.983323,2909.951957,0.923482,0.781718,False
9,Pando,Nicolás Suárez,Bella Flor,C4,3421.0,3344.896796,3290.412970,-0.204519,-0.109485,False


In [20]:
n_sign = ctrl["sign_change"].sum()
print(f"{n_sign} municipalité(s) avec changement de signe du taux 2024-2035 -> 2035-2050 :")
ctrl[ctrl["sign_change"]][KEYS + ["taux_2435", "taux_3550"]]

3 municipalité(s) avec changement de signe du taux 2024-2035 -> 2035-2050 :


,DEPARTAMENTO,PROVINCIA,MUNICIPIO/TIOC,taux_2435,taux_3550
3,Beni,General José Ballivián,Reyes,-0.414187,0.087365
18,Pando,Federico Román,Nueva Esperanza,-0.259524,0.083406
19,Pando,Federico Román,Villa Nueva,-0.131086,0.078751


In [21]:
cluster_ctrl = ctrl.groupby("cluster_label")[["pop_2024", "pop_2035", "pop_2050"]].sum()
cluster_ctrl["taux_2435"] = 100 * np.log(cluster_ctrl["pop_2035"] / cluster_ctrl["pop_2024"]) / 11
cluster_ctrl["taux_3550"] = 100 * np.log(cluster_ctrl["pop_2050"] / cluster_ctrl["pop_2035"]) / 15
cluster_ctrl

,pop_2024,pop_2035,pop_2050,taux_2435,taux_3550
cluster_label,,,,,
C1,41377.0,43743.014753,46445.023772,0.505516,0.399583
C2,2338.0,2587.983323,2909.951957,0.923482,0.781718
C3,159706.0,175343.831306,192393.831564,0.849220,0.618638
C4,62188.0,69245.252957,74970.417426,0.977205,0.529593
C5,51908.0,58615.103461,64557.438773,1.104723,0.643753


## Étape 4 — vérification de signe et backtest de la règle rho

### 4.1 Vérification de signe — série INE 2024-2034 (Reyes, Nueva Esperanza, Villa Nueva)

Question précise : le taux INE 2033→2034 est-il positif ou négatif pour ces trois
municipalités ? Positif -> le changement de signe de l'étape 3.4 vient de l'INE, pas de notre
règle. Négatif -> bug, arrêt.

In [22]:
check_munis = ["Reyes", "Nueva Esperanza", "Villa Nueva"]
sub_m = m[m["MUNICIPIO/TIOC"].isin(check_munis)]
sub_m[KEYS + [str(y) for y in YEARS_HIST]]

,DEPARTAMENTO,PROVINCIA,MUNICIPIO/TIOC,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034
3,Beni,General José Ballivián,Reyes,12695.0,12539.0,12415.0,12312.0,12238.0,12176.0,12124.0,12113.0,12102.0,12100.0,12115.0
18,Pando,Federico Román,Nueva Esperanza,1743.0,1724.0,1718.0,1705.0,1694.0,1692.0,1694.0,1687.0,1696.0,1690.0,1692.0
19,Pando,Federico Román,Villa Nueva,2730.0,2728.0,2720.0,2708.0,2699.0,2702.0,2691.0,2682.0,2692.0,2685.0,2688.0


In [23]:
rate_rows = []
for _, row in sub_m.iterrows():
    for y in range(2025, 2035):
        rate_rows.append(dict(municipio=row["MUNICIPIO/TIOC"], annee=y,
                               taux=100 * math.log(row[str(y)] / row[str(y - 1)])))
rate_df = pd.DataFrame(rate_rows).pivot(index="annee", columns="municipio", values="taux")

taux_3334 = rate_df.loc[2034]
assert (taux_3334 >= 0).all(), f"STOP: taux INE 2033->2034 négatif détecté :\n{taux_3334}"
print("Taux INE 2033->2034 (%/an) :")
print(taux_3334)
print("Positif pour les trois -> le changement de signe de l'étape 3.4 vient de la trajectoire "
      "INE elle-même (creux 2025-2033 puis rebond), pas d'un bug de la règle rho.")
rate_df

Taux INE 2033->2034 (%/an) :
municipio
Nueva Esperanza    0.118273
Reyes              0.123890
Villa Nueva        0.111669
Name: 2034, dtype: float64
Positif pour les trois -> le changement de signe de l'étape 3.4 vient de la trajectoire INE elle-même (creux 2025-2033 puis rebond), pas d'un bug de la règle rho.


municipio,Nueva Esperanza,Reyes,Villa Nueva
annee,,,
2025,-1.096059,-1.236443,-0.073287
2026,-0.348635,-0.993837,-0.293686
2027,-0.759571,-0.833102,-0.442153
2028,-0.647251,-0.602853,-0.332902
2029,-0.118134,-0.507906,0.111091
2030,0.118134,-0.427984,-0.407937
2031,-0.414079,-0.090770,-0.335009
2032,0.532073,-0.090853,0.372163
2033,-0.354401,-0.016528,-0.260368


### 4.2 Backtest de la règle rho, 2012→2024

Analogue exact de l'extrapolation 2034→2050, calé 12 ans plus tôt, vérifié contre le Censo 2024 :
- `taux_i(2012) = 100·ln(P_i(2012)/P_i(2001))/11` (observé, `CSV_final`)
- `rho_obs(t) = taux_national_INE(t) / moyenne(taux_national_INE sur 2010-2012)`, t=2013..2024
- `taux_i(t) = taux_i(2012)·rho_obs(t)`, chaînage annuel depuis `pop_2012_censo`

In [24]:
taux_i_2012 = 100 * np.log(base["pop_2012_censo"] / base["pop_2001_censo"]) / 11

ref_obs = rate_nat.loc[[2010, 2011, 2012]].mean()
YEARS_BT = list(range(2013, 2025))
rho_obs = {y: rate_nat.loc[y] / ref_obs for y in YEARS_BT}

pop_bt = pd.DataFrame(index=base.index, columns=range(2012, 2025), dtype=float)
pop_bt[2012] = base["pop_2012_censo"]
prev = pop_bt[2012].copy()
for y in YEARS_BT:
    taux_y = taux_i_2012 * rho_obs[y]
    prev = prev * (1 + taux_y / 100)
    pop_bt[y] = prev

retro["rho"] = pop_bt[2024].values
retro["APE_rho"] = 100 * (retro["rho"] - retro["P2024_obs"]).abs() / retro["P2024_obs"]
MODELS6 = MODELS + ["rho"]

retro[["municipio", "P2024_obs", "rho", "APE_rho"]]

,municipio,P2024_obs,rho,APE_rho
0,Ixiamas,11330.0,11690.378041,3.180742
1,Riberalta,107816.0,97666.896569,9.413356
2,Guayaramerín,40130.0,40940.671515,2.020113
3,Reyes,11284.0,14488.533118,28.398911
4,Santa Rosa,10953.0,9585.481921,12.485329
5,Exaltación,7810.0,5649.734616,27.660248
6,Cobija,51908.0,73680.340733,41.944095
7,Porvenir,9096.0,13200.469997,45.123901
8,Bolpebra,2338.0,3145.206961,34.525533
9,Bella Flor,3421.0,5102.924904,49.164715


In [25]:
weights = retro["P2024_obs"]
weighted_mean = {}
for mo in MODELS6:
    s = retro[f"APE_{mo}"]
    valid = s.notna()
    weighted_mean[mo] = (s[valid] * weights[valid]).sum() / weights[valid].sum()

unweighted_rank6 = retro[[f"APE_{mo}" for mo in MODELS6]].mean(skipna=True).rename("unweighted_mean_APE")
unweighted_rank6 = unweighted_rank6.sort_values().rename_axis("modele").reset_index()
unweighted_rank6["modele"] = unweighted_rank6["modele"].str.replace("APE_", "")

weighted_rank6 = pd.Series(weighted_mean, name="weighted_mean_APE").sort_values().rename_axis("modele").reset_index()

classement6_side_by_side = pd.concat([unweighted_rank6, weighted_rank6], axis=1, keys=["non pondéré", "pondéré"])
classement6_side_by_side.index = classement6_side_by_side.index + 1
classement6_side_by_side

non pondéré                                       pondéré  \
                     modele unweighted_mean_APE                    modele   
1  logistique_proportionnel           15.416145  logistique_proportionnel   
2           share_of_growth           19.956239           share_of_growth   
3                logistique           28.621508                logistique   
4                  lineaire           29.414116                  lineaire   
5                       rho           43.125598                       rho   
6               exponentiel           88.717782               exponentiel   

                     
  weighted_mean_APE  
1          9.629062  
2         12.425564  
3         15.695509  
4         16.395944  
5         23.113221  
6         43.876790

### 4.3 Tableau cluster complet, six modèles

Écart en % entre la somme des prédictions 2024 et la population observée, par cluster.
Colonnes triées par erreur moyenne croissante (meilleur modèle à gauche).

In [26]:
cluster_obs = retro.groupby("cluster_label")["P2024_obs"].sum()

tab6 = {}
for mo in MODELS6:
    valid = retro[mo].notna()
    pred_by_cluster = retro[valid].groupby("cluster_label")[mo].sum()
    tab6[mo] = 100 * (pred_by_cluster - cluster_obs) / cluster_obs

cluster_tab6 = pd.DataFrame(tab6).reindex(sorted(cluster_obs.index))
cluster_tab6 = cluster_tab6[unweighted_rank6["modele"]].round(1)
cluster_tab6

,logistique_proportionnel,share_of_growth,logistique,lineaire,rho,exponentiel
cluster_label,,,,,,
C1,-17.0,-3.7,-12.0,1.2,0.1,7.6
C2,9.4,14.7,30.4,31.0,34.5,68.2
C3,-7.2,-8.2,-3.6,-3.5,-5.3,1.0
C4,2.9,11.5,28.5,29.6,50.3,115.0
C5,5.9,13.7,31.0,31.8,41.9,87.7


### 4.4 Classement médiane, six modèles

Même `stats` que l'étape 2 (mean_APE, median_APE, CV_error), étendu à `rho`, trié une fois par
moyenne et une fois par médiane.

In [27]:
stats6 = pd.DataFrame({
    mo: {
        "mean_APE": retro[f"APE_{mo}"].mean(skipna=True),
        "median_APE": retro[f"APE_{mo}"].median(skipna=True),
        "CV_error": retro[f"APE_{mo}"].std(skipna=True) / retro[f"APE_{mo}"].mean(skipna=True),
    } for mo in MODELS6
}).T

classement6_mean = stats6.sort_values("mean_APE").reset_index().rename(columns={"index": "modele"})
classement6_mean.index = classement6_mean.index + 1
classement6_mean

,modele,mean_APE,median_APE,CV_error
1,logistique_proportionnel,15.416145,11.915060,0.806110
2,share_of_growth,19.956239,14.614375,0.840133
3,logistique,28.621508,22.351874,0.954559
4,lineaire,29.414116,30.322430,0.932811
5,rho,43.125598,27.660248,1.062579
6,exponentiel,88.717782,45.582358,1.070447


In [28]:
classement6_median = stats6.sort_values("median_APE").reset_index().rename(columns={"index": "modele"})
classement6_median.index = classement6_median.index + 1
classement6_median

,modele,mean_APE,median_APE,CV_error
1,logistique_proportionnel,15.416145,11.915060,0.806110
2,share_of_growth,19.956239,14.614375,0.840133
3,logistique,28.621508,22.351874,0.954559
4,rho,43.125598,27.660248,1.062579
5,lineaire,29.414116,30.322430,0.932811
6,exponentiel,88.717782,45.582358,1.070447


## Étape 5 — intervalle d'application et test restreint

### 5.2 Intervalle d'application — taux INE 2033→2034 (21 municipalités)

À mesurer avant tout le reste : c'est l'intervalle des taux de base que `rho` reçoit
réellement en entrée du backtest (via `taux_i(2012)` dans la règle, structurellement
analogue à `taux_i(2034)` dans la projection principale).

In [29]:
taux_2034 = 100 * np.log(m["2034"] / m["2033"])
interval_table = pd.DataFrame({"municipio": m["MUNICIPIO/TIOC"], "taux_2033_2034": taux_2034.values}) \
    .sort_values("taux_2033_2034").reset_index(drop=True)

INTERVAL_MIN, INTERVAL_MAX = taux_2034.min(), taux_2034.max()
print(f"min={INTERVAL_MIN:.4f}  max={INTERVAL_MAX:.4f}  médiane={taux_2034.median():.4f}")
interval_table

min=-0.1551  max=1.1131  médiane=0.8290


,municipio,taux_2033_2034
0,Bella Flor,-0.155079
1,Villa Nueva,0.111669
2,Nueva Esperanza,0.118273
3,Reyes,0.123890
4,Ixiamas,0.336520
5,Ingavi,0.633314
6,Santa Rosa,0.667734
7,Filadelfia,0.798367
8,San Lorenzo,0.804227
9,Guayaramerín,0.825000


### 5.3 Test restreint

Pré-spécifié par l'intervalle mesuré en 5.2, pas par les résultats : ne garde que les
municipalités dont le taux 2001-2012 (`retro["taux_0112"]`, étape 3.3) tombe dans
`[INTERVAL_MIN, INTERVAL_MAX]`.

In [30]:
in_range = (retro["taux_0112"] >= INTERVAL_MIN) & (retro["taux_0112"] <= INTERVAL_MAX)
n_retenues = int(in_range.sum())
print(f"{n_retenues} municipalité(s) retenue(s) sur 21.")
if n_retenues < 8:
    print("Moins de 8 municipalités retenues : échantillon restreint, chiffres fournis sans interprétation.")

retro_r = retro[in_range]
retro_r[["municipio", "taux_0112"] + [f"APE_{mo}" for mo in MODELS6]]

2 municipalité(s) retenue(s) sur 21.
Moins de 8 municipalités retenues : échantillon restreint, chiffres fournis sans interprétation.


,municipio,taux_0112,APE_lineaire,APE_exponentiel,APE_share_of_growth,APE_logistique,APE_logistique_proportionnel,APE_rho
2,Guayaramerín,0.232099,2.893324,2.967831,1.846832,2.893213,2.749925,2.020113
4,Santa Rosa,0.381423,11.314459,11.143802,12.760317,11.314866,11.627657,12.485329


In [31]:
stats_r = pd.DataFrame({
    mo: {
        "n": retro_r[f"APE_{mo}"].notna().sum(),
        "mean_APE": retro_r[f"APE_{mo}"].mean(skipna=True),
        "median_APE": retro_r[f"APE_{mo}"].median(skipna=True),
    } for mo in MODELS6
}).T

w = retro_r["P2024_obs"]
weighted_r = {}
for mo in MODELS6:
    s = retro_r[f"APE_{mo}"]
    valid = s.notna()
    weighted_r[mo] = (s[valid] * w[valid]).sum() / w[valid].sum() if valid.sum() else np.nan
stats_r["weighted_mean_APE"] = pd.Series(weighted_r)
stats_r

,n,mean_APE,median_APE,weighted_mean_APE
lineaire,2.0,7.103892,7.103892,4.698948
exponentiel,2.0,7.055816,7.055816,4.720888
share_of_growth,2.0,7.303575,7.303575,4.186856
logistique,2.0,7.104040,7.104040,4.698948
logistique_proportionnel,2.0,7.188791,7.188791,4.653451
rho,2.0,7.252721,7.252721,4.264021


In [32]:
cluster_obs_r = retro_r.groupby("cluster_label")["P2024_obs"].sum()
tab_r = {}
for mo in MODELS6:
    valid = retro_r[mo].notna()
    pred_c = retro_r[valid].groupby("cluster_label")[mo].sum()
    tab_r[mo] = 100 * (pred_c - cluster_obs_r.reindex(pred_c.index)) / cluster_obs_r.reindex(pred_c.index)
pd.DataFrame(tab_r)

,lineaire,exponentiel,share_of_growth,logistique,logistique_proportionnel,rho
cluster_label,,,,,,
C1,-11.314459,-11.143802,-12.760317,-11.314866,-11.627657,-12.485329
C3,2.893324,2.967831,1.846832,2.893213,2.749925,2.020113


### 5.4 Amplification — corrélation APE vs taux de base 2001-2012

Sur le backtest complet (21 municipalités, 20 pour `logistique_proportionnel` — Exaltación
exclue, échec de domaine étape 2).

In [33]:
from scipy import stats as spstats

corr_rows = []
for mo in ["rho", "logistique_proportionnel"]:
    sub = retro.dropna(subset=[f"APE_{mo}"])
    pear = spstats.pearsonr(sub["taux_0112"], sub[f"APE_{mo}"])
    spear = spstats.spearmanr(sub["taux_0112"], sub[f"APE_{mo}"])
    corr_rows.append(dict(modele=mo, n=len(sub), pearson_r=pear[0], pearson_p=pear[1],
                           spearman_rho=spear.correlation, spearman_p=spear.pvalue))
pd.DataFrame(corr_rows)

,modele,n,pearson_r,pearson_p,spearman_rho,spearman_p
0,rho,21,0.773893,0.000039,0.675325,0.000782
1,logistique_proportionnel,20,0.220079,0.351154,0.150376,0.526855


## Étape 6 — asymptotes multiplicatives et test en régime d'application

### 6.1 Réparation Exaltación — jeu d'asymptotes K (INE §7.2.5.d)

L'échec de domaine (étape 2) vient du choix `K = 2·P(2012)` : Exaltación est la seule
municipalité en décroissance 2001-2012 (`P(2001) > P(2012)`), une asymptote *au-dessus* de la
population courante n'a pas de solution valide dans ce cas. Jeu de facteurs multiplicatifs
essayés dans l'ordre `[2.0, 0.5]` — supérieur (croissance) d'abord, inférieur (déclin) en
repli — appliqué à `P(2012)` pour `logistique`, et à `K_région·part_2012` puis en repli à
`0.5·P(2012)` pour `logistique_proportionnel`. Les 20 autres municipalités retombent sur le
facteur 2.0 : résultats inchangés.

In [34]:
K_FACTORS = [2.0, 0.5]

def solve_with_factors(P0, P1, h1, factors, K_base):
    for f in factors:
        K = f * K_base
        ab = solve_ab(K, P0, P1, h1)
        if ab is not None:
            return K, f, ab
    return None, None, None

f_log_used, f_logp_used = [], []
for i, row in base.iterrows():
    P0, P1 = row["pop_2001_censo"], row["pop_2012_censo"]

    K_log, f_log, ab = solve_with_factors(P0, P1, 11, K_FACTORS, P1)
    retro.loc[i, "logistique"] = logistic(K_log, ab[0], ab[1], 2024, 2001)
    f_log_used.append(f_log)

    K_base_prop = K_region * (P1 / P1r)
    ab2 = solve_ab(K_base_prop, P0, P1, 11)
    if ab2 is not None:
        K_logp, f_logp = K_base_prop, "region-share"
    else:
        K_logp, f_logp, ab2 = solve_with_factors(P0, P1, 11, [0.5], P1)
    retro.loc[i, "logistique_proportionnel"] = logistic(K_logp, ab2[0], ab2[1], 2024, 2001)
    f_logp_used.append(f_logp)

retro["f_log"] = f_log_used
retro["f_logp"] = f_logp_used
for mo in ["logistique", "logistique_proportionnel"]:
    retro[f"APE_{mo}"] = 100 * (retro[mo] - retro["P2024_obs"]).abs() / retro["P2024_obs"]

print("NaN restants (logistique, logistique_proportionnel):", retro[["logistique", "logistique_proportionnel"]].isna().sum().sum())
retro[["municipio", "f_log", "f_logp", "logistique", "logistique_proportionnel", "APE_logistique", "APE_logistique_proportionnel"]]

NaN restants (logistique, logistique_proportionnel): 0


,municipio,f_log,f_logp,logistique,logistique_proportionnel,APE_logistique,APE_logistique_proportionnel
0,Ixiamas,2.0,region-share,11722.548752,10189.929673,3.464684,10.062404
1,Riberalta,2.0,region-share,100979.924298,97516.470191,6.340502,9.552877
2,Guayaramerín,2.0,region-share,41291.046413,41233.545064,2.893213,2.749925
3,Reyes,2.0,region-share,14977.530231,14474.157298,32.732455,28.271511
4,Santa Rosa,2.0,region-share,9713.682686,9679.422755,11.314866,11.627657
5,Exaltación,0.5,0.5,5558.196615,5558.196615,28.832310,28.832310
6,Cobija,2.0,region-share,67982.578687,54972.656626,30.967440,5.904016
7,Porvenir,2.0,region-share,12040.010186,9669.027985,32.365987,6.299780
8,Bolpebra,2.0,region-share,3049.628165,2557.484106,30.437475,9.387686
9,Bella Flor,2.0,region-share,5072.482328,4361.911548,48.274841,27.503991


### Tableau cluster complet, six modèles, sans exclusion

In [35]:
cluster_obs = retro.groupby("cluster_label")["P2024_obs"].sum()
tab6_full = {mo: 100 * (retro.groupby("cluster_label")[mo].sum() - cluster_obs) / cluster_obs for mo in MODELS6}
cluster_tab6_full = pd.DataFrame(tab6_full).reindex(sorted(cluster_obs.index))
cluster_tab6_full = cluster_tab6_full[unweighted_rank6["modele"]].round(1)
cluster_tab6_full

,logistique_proportionnel,share_of_growth,logistique,lineaire,rho,exponentiel
cluster_label,,,,,,
C1,-3.6,-3.7,1.4,1.2,0.1,7.6
C2,9.4,14.7,30.4,31.0,34.5,68.2
C3,-7.2,-8.2,-3.6,-3.5,-5.3,1.0
C4,2.9,11.5,28.5,29.6,50.3,115.0
C5,5.9,13.7,31.0,31.8,41.9,87.7


### 6.2 Test en régime d'application — rho vs logistique proportionnel sur série INE 2024-2034

Mesure d'écart entre deux règles d'extension candidates, dans le régime où elles s'appliquent
réellement (projection 2024→2050 à partir de la série INE municipale) — **pas une validation
contre un observé** : la population 2050 n'est pas mesurée, seulement projetée par l'INE au
niveau national.

- **rho** : projection principale des étapes 1-5 (`pop`), inchangée.
- **logistique proportionnel (INE)** : calibré sur `P_INE(2024)` et `P_INE(2034)` de chaque
  municipalité ; `K_national` résolu exactement sur 3 points (national INE 2024, 2034, 2050,
  `POBLACIÓN AL 01 DE ENERO`) ; `K_i = K_national·P_INE,i(2034)/P_national(2034)` ; repli à
  `0.5·P_INE,i(2034)` (même jeu d'asymptotes qu'en 6.1) si domaine invalide.

In [36]:
pop_nat = nat.set_index("AÑO")["POBLACIÓN AL 01 DE ENERO"]

nat_2024 = float(pop_nat.loc[2024])
nat_2034 = float(pop_nat.loc[2034])
nat_2050 = float(pop_nat.loc[2050])

def nat_resid(K):
    a, b = solve_ab(K, nat_2024, nat_2034, 10)
    return logistic(K, a, b, 2050, 2024) - nat_2050

K_nat = brentq(nat_resid, nat_2050 * 1.0001, nat_2050 * 50)
print(f"K_national = {K_nat:.0f}  (nat_2024={nat_2024:.0f}, nat_2034={nat_2034:.0f}, nat_2050={nat_2050:.0f})")

rows_app = []
for _, row in m.iterrows():
    P0i, P1i = row["2024"], row["2034"]
    K_share = K_nat * (P1i / nat_2034)
    ab = solve_ab(K_share, P0i, P1i, 10)
    if ab is not None:
        K_used, factor_used = K_share, "share"
    else:
        K_used = 0.5 * P1i
        ab = solve_ab(K_used, P0i, P1i, 10)
        factor_used = 0.5
    rows_app.append(dict(municipio=row["MUNICIPIO/TIOC"], factor=factor_used,
                          logprop_2035=logistic(K_used, ab[0], ab[1], 2035, 2024),
                          logprop_2050=logistic(K_used, ab[0], ab[1], 2050, 2024)))

app = pd.DataFrame(rows_app)
app["cluster_label"] = base_cl["cluster_label"].values
app["rho_2035"] = pop[2035].values
app["rho_2050"] = pop[2050].values
print("municipalités en repli (facteur 0.5) :", app.loc[app["factor"] == 0.5, "municipio"].tolist())
app

K_national = 12480029  (nat_2024=11916453, nat_2034=12157674, nat_2050=12350297)
municipalités en repli (facteur 0.5) : ['Reyes', 'Bella Flor', 'Nueva Esperanza', 'Villa Nueva']


,municipio,factor,logprop_2035,logprop_2050,cluster_label,rho_2035,rho_2050
0,Ixiamas,share,14038.205598,14322.189824,C1,12296.072825,12741.251338
1,Riberalta,share,133785.911541,136527.322912,C3,119725.131655,131683.701488
2,Guayaramerín,share,48436.465818,49327.282019,C3,42182.904652,46017.859874
3,Reyes,0.5,12061.237384,11333.189690,C1,10781.430736,10923.648726
4,Santa Rosa,share,13316.844014,13587.226983,C1,12003.436352,13273.440312
5,Exaltación,share,9712.682236,9911.604229,C1,8662.074839,9506.683396
6,Cobija,share,67073.607765,68455.401980,C5,58615.103461,64557.438773
7,Porvenir,share,11348.020598,11581.693409,C4,10247.773343,11286.266084
8,Bolpebra,share,2901.719910,2961.009893,C2,2587.983323,2909.951957
9,Bella Flor,0.5,3858.077000,3744.830176,C4,3344.896796,3290.412970


Population par cluster selon chaque règle, et écart relatif entre les deux, pour 2035 et 2050.

In [37]:
cluster_app = app.groupby("cluster_label")[["rho_2035", "logprop_2035", "rho_2050", "logprop_2050"]].sum()
cluster_app["ecart_2035_pct"] = 100 * (cluster_app["logprop_2035"] - cluster_app["rho_2035"]) / cluster_app["rho_2035"]
cluster_app["ecart_2050_pct"] = 100 * (cluster_app["logprop_2050"] - cluster_app["rho_2050"]) / cluster_app["rho_2050"]
cluster_app.round(1)

,rho_2035,logprop_2035,rho_2050,logprop_2050,ecart_2035_pct,ecart_2050_pct
cluster_label,,,,,,
C1,43743.0,49129.0,46445.0,49154.2,12.3,5.8
C2,2588.0,2901.7,2910.0,2961.0,12.1,1.8
C3,175343.8,197451.7,192393.8,201398.2,12.6,4.7
C4,69245.3,77009.9,74970.4,78186.4,11.2,4.3
C5,58615.1,67073.6,64557.4,68455.4,14.4,6.0


### 6.3

Projection principale des étapes 1-5 non modifiée. Ce tableau documente l'écart entre les deux
règles d'extension candidates pour trancher — le choix n'est pas fait ici.

## Étape 7 — diagnostic 6.2 et réparation (calage, continuité, renormalisation)

### 7.1 Diagnostic — sur quoi la logistique de 6.2 est-elle calée ?

Cellule `f6f79c81` (6.2) : `P0i, P1i = row["2024"], row["2034"]` — ces valeurs viennent de `m`,
donc de la **série INE 2024-2034**, pas des recensements. La continuité en 2034 y est déjà
quasi exacte par construction (vérifié ci-dessous).

L'écart 1,952 %/an vs 0,873 %/an ne vient donc pas du calage mais du **niveau de référence** :
la somme INE 2024 des 21 municipalités (`m["2024"].sum()`) est ~13 % au-dessus du Censo 2024
réel (`pop_region[2024]`, la même base que `rho`). Le pipeline logistique-de-parts de 6.2 reste
ancré sur ce total INE gonflé du début à la fin ; comparé au Censo 2024 réel ça reproduit
exactement 1,952 %/an — pas un défaut de calage, un défaut de renormalisation (point 7.2c).

In [38]:
sum_2024_INE = m["2024"].sum()
ecart_base_pct = 100 * (sum_2024_INE - pop_region[2024]) / pop_region[2024]
print(f"somme INE 2024 (21 municipalités)  = {sum_2024_INE:.0f}")
print(f"Censo 2024 réel (pop_region[2024]) = {pop_region[2024]:.0f}")
print(f"écart de base                      = {ecart_base_pct:.2f} %")

taux_mixte = 100 * math.log(app["logprop_2035"].sum() / pop_region[2024]) / 11
print(f"taux mixte (prédiction 6.2 ancrée INE / Censo 2024 réel) 2024-2035 = {taux_mixte:.3f} %/an")

somme INE 2024 (21 municipalités)  = 359200
Censo 2024 réel (pop_region[2024]) = 317517
écart de base                      = 13.13 %
taux mixte (prédiction 6.2 ancrée INE / Censo 2024 réel) 2024-2035 = 1.952 %/an


### 7.2 Réimplémentation — trois conditions, chacune vérifiée par assertion

**a. Calage** : uniquement sur `m["2024"]`/`m["2034"]` (INE), jamais sur `base` (censo) — vérifié
ligne à ligne dans la boucle de calibration.
**b. Continuité** : `|courbe(2034) − P_INE(2034)| / P_INE(2034) < 0,1 %` pour les 21 municipalités,
assertion bloquante.
**c. Somme** : renormalisation post-projection vers un total de contrôle.

In [39]:
nat_2024 = float(pop_nat.loc[2024])
nat_2034 = float(pop_nat.loc[2034])
nat_2050 = float(pop_nat.loc[2050])
a_nat, b_nat = solve_ab(K_nat, nat_2024, nat_2034, 10)
assert abs(logistic(K_nat, a_nat, b_nat, 2050, 2024) - nat_2050) / nat_2050 < 1e-9

rows_app7 = []
for i, row in m.iterrows():
    P0i, P1i = float(row["2024"]), float(row["2034"])
    assert P0i == float(m.loc[i, "2024"]) and P1i == float(m.loc[i, "2034"])  # (a) calage INE, jamais censo

    K_share = K_nat * (P1i / nat_2034)
    ab = solve_ab(K_share, P0i, P1i, 10)
    if ab is not None:
        K_used, factor_used = K_share, "share"
    else:
        K_used = 0.5 * P1i
        ab = solve_ab(K_used, P0i, P1i, 10)
        factor_used = 0.5
    a_i, b_i = ab

    pred_2034_check = logistic(K_used, a_i, b_i, 2034, 2024)
    rel_err_2034 = abs(pred_2034_check - P1i) / P1i
    rows_app7.append(dict(municipio=row["MUNICIPIO/TIOC"], factor=factor_used, rel_err_2034=rel_err_2034,
                           raw_2035=logistic(K_used, a_i, b_i, 2035, 2024),
                           raw_2050=logistic(K_used, a_i, b_i, 2050, 2024)))

app7 = pd.DataFrame(rows_app7)
max_err_2034 = app7["rel_err_2034"].max()
print(f"écart max continuité 2034 = {max_err_2034:.2e} %")
assert max_err_2034 < 0.001, f"STOP continuité: {max_err_2034}"  # (b) < 0.1 %
print("continuité OK (< 0.1 %) pour les 21 municipalités")

écart max continuité 2034 = 1.52e-16 %
continuité OK (< 0.1 %) pour les 21 municipalités


**Total de contrôle (c)** : population nationale INE réelle (`POBLACIÓN AL 01 DE ENERO`, pas un
modèle) à l'année cible. **Passage national → région** : part du Censo 2024 réel de la région
dans la nation (`pop_region[2024] / nat_2024`), maintenue constante — `contrôle_région(t) =
pop_nat(t) · part_2024`. Les parts municipales brutes (INE, non renormalisées) sont ensuite
mises à l'échelle pour que leur somme égale ce contrôle, à structure relative inchangée.

In [40]:
region_share_2024 = pop_region[2024] / nat_2024
print(f"part région/nation Censo 2024 = {region_share_2024:.6f}")

app7["cluster_label"] = base_cl["cluster_label"].values
control = {}
for t in (2035, 2050):
    control[t] = float(pop_nat.loc[t]) * region_share_2024
    raw_sum = app7[f"raw_{t}"].sum()
    scale = control[t] / raw_sum
    app7[f"logprop_{t}"] = app7[f"raw_{t}"] * scale
    renorm_sum = app7[f"logprop_{t}"].sum()
    assert abs(renorm_sum - control[t]) / control[t] < 1e-9, f"STOP somme {t}"  # (c)
    print(f"t={t}  contrôle={control[t]:.0f}  brut={raw_sum:.0f}  échelle={scale:.4f}  après={renorm_sum:.0f}")

app7["rho_2035"] = pop[2035].values
app7["rho_2050"] = pop[2050].values
app7[["municipio", "factor", "rel_err_2034", "logprop_2035", "logprop_2050"]]

part région/nation Censo 2024 = 0.026645
t=2035  contrôle=324373  brut=393566  échelle=0.8242  après=324373
t=2050  contrôle=329077  brut=400155  échelle=0.8224  après=329077


,municipio,factor,rel_err_2034,logprop_2035,logprop_2050
0,Ixiamas,share,0.000000e+00,11570.143864,11778.183777
1,Riberalta,share,0.000000e+00,110264.964611,112276.399041
2,Guayaramerín,share,1.505786e-16,39920.834173,40565.430285
3,Reyes,0.5,0.000000e+00,9940.747108,9320.110443
4,Santa Rosa,share,1.370753e-16,10975.605107,11173.770100
5,Exaltación,share,0.000000e+00,8005.092246,8151.036787
6,Cobija,share,0.000000e+00,55281.373811,56295.881771
7,Porvenir,share,0.000000e+00,9352.921210,9524.473219
8,Bolpebra,share,0.000000e+00,2391.567539,2435.054912
9,Bella Flor,0.5,0.000000e+00,3179.787161,3079.647636


### 7.3 Tableau 6.2 corrigé

In [41]:
cluster_app7 = app7.groupby("cluster_label")[["rho_2035", "logprop_2035", "rho_2050", "logprop_2050"]].sum()
cluster_app7["ecart_2035_pct"] = 100 * (cluster_app7["logprop_2035"] - cluster_app7["rho_2035"]) / cluster_app7["rho_2035"]
cluster_app7["ecart_2050_pct"] = 100 * (cluster_app7["logprop_2050"] - cluster_app7["rho_2050"]) / cluster_app7["rho_2050"]
cluster_app7.round(1)

,rho_2035,logprop_2035,rho_2050,logprop_2050,ecart_2035_pct,ecart_2050_pct
cluster_label,,,,,,
C1,43743.0,40491.6,46445.0,40423.1,-7.4,-13.0
C2,2588.0,2391.6,2910.0,2435.1,-7.6,-16.3
C3,175343.8,162737.7,192393.8,165624.4,-7.2,-13.9
C4,69245.3,63470.8,74970.4,64298.4,-8.3,-14.2
C5,58615.1,55281.4,64557.4,56295.9,-5.7,-12.8


Lignes de contrôle, au total régional : taux annuel implicite 2024-2035 et 2035-2050 pour
chaque règle, et part de la région dans la population nationale INE en 2024, 2035, 2050.

In [42]:
rho_2024_tot, rho_2035_tot, rho_2050_tot = pop_region[2024], cluster_app7["rho_2035"].sum(), cluster_app7["rho_2050"].sum()
lp_2024_tot, lp_2035_tot, lp_2050_tot = pop_region[2024], cluster_app7["logprop_2035"].sum(), cluster_app7["logprop_2050"].sum()

controle = pd.DataFrame({
    "rho": {
        "taux_2024_2035": 100 * math.log(rho_2035_tot / rho_2024_tot) / 11,
        "taux_2035_2050": 100 * math.log(rho_2050_tot / rho_2035_tot) / 15,
        "part_nation_2024": pop_region[2024] / nat_2024,
        "part_nation_2035": rho_2035_tot / float(pop_nat.loc[2035]),
        "part_nation_2050": rho_2050_tot / float(pop_nat.loc[2050]),
    },
    "logistique_proportionnel": {
        "taux_2024_2035": 100 * math.log(lp_2035_tot / lp_2024_tot) / 11,
        "taux_2035_2050": 100 * math.log(lp_2050_tot / lp_2035_tot) / 15,
        "part_nation_2024": pop_region[2024] / nat_2024,
        "part_nation_2035": lp_2035_tot / float(pop_nat.loc[2035]),
        "part_nation_2050": lp_2050_tot / float(pop_nat.loc[2050]),
    },
})
controle

,rho,logistique_proportionnel
taux_2024_2035,0.873390,0.194206
taux_2035_2050,0.579474,0.095983
part_nation_2024,0.026645,0.026645
part_nation_2035,0.028712,0.026645
part_nation_2050,0.030872,0.026645


### 7.4

Renormalisée au total national INE, la règle logistique proportionnelle projette une région
**plus lente** que `rho` (part constante dans la nation par construction, contre une part
croissante pour `rho`) — inversion complète par rapport au tableau 6.2 brut. Projection
principale des étapes 1-5 non modifiée ; le choix entre les deux règles reste ouvert.

## Étape 8 — vérification du +13,13 %, backtest agrégé, architecture combinée

### 8.1 Vérification du +13,13 % — double comptage ?

Contrôle du merge (clé dupliquée côté INE ?), recherche de TIOC recouvrant une municipalité déjà
comptée dans nos 3 départements, puis rapport INE/Censo municipalité par municipalité.

In [43]:
print("doublons de clé dans ine_muni :", ine_muni.duplicated(subset=KEYS).sum())
print("len(m) :", len(m), "(doit être 21 — sinon fan-out au merge)")

target_depts = base["DEPARTAMENTO"].unique().tolist()
zone = ine_muni_raw[ine_muni_raw["DEPARTAMENTO"].isin(target_depts) & (ine_muni_raw["NIVEL"] == "Municipio/TIOC")]
name_counts = zone.groupby(["DEPARTAMENTO", "MUNICIPIO/TIOC"]).size()
dups = name_counts[name_counts > 1]
print("noms dupliqués (même département) dans nos 3 départements :", dict(dups) if len(dups) else "aucun")

suspects = zone[zone["MUNICIPIO/TIOC"].astype(str).str.contains(
    "TCO|T\\.C\\.O|Territorio|Ind[ií]gena", case=False, regex=True, na=False)]
print("lignes avec marqueur territoire indigène dans nos 3 départements :")
print(suspects[["DEPARTAMENTO", "PROVINCIA", "MUNICIPIO/TIOC"]] if len(suspects) else "aucune — pas de recouvrement TIOC dans la zone")

doublons de clé dans ine_muni : 0
len(m) : 21 (doit être 21 — sinon fan-out au merge)
noms dupliqués (même département) dans nos 3 départements : aucun
lignes avec marqueur territoire indigène dans nos 3 départements :
    DEPARTAMENTO PROVINCIA                        MUNICIPIO/TIOC
443         Beni    Iténez       Territorio Indígena Multiétnico
444         Beni    Iténez  TIOC-Territorio Indígena Multiétnico


In [44]:
ratio_table = m[KEYS + ["pop_2024_censo", "2024"]].copy()
ratio_table["ratio_pct"] = 100 * (ratio_table["2024"] / ratio_table["pop_2024_censo"] - 1)
ratio_table = ratio_table.sort_values("ratio_pct").reset_index(drop=True)

flag = ratio_table[(ratio_table["ratio_pct"] > 25) | (ratio_table["ratio_pct"] < 0)]
print(f"{len(flag)} municipalité(s) hors [0 %, +25 %] :", flag["MUNICIPIO/TIOC"].tolist() if len(flag) else "aucune")
print(f"écart global : {100*(ratio_table['2024'].sum()/ratio_table['pop_2024_censo'].sum()-1):.2f} %")
ratio_table

0 municipalité(s) hors [0 %, +25 %] : aucune
écart global : 13.13 %


,DEPARTAMENTO,PROVINCIA,MUNICIPIO/TIOC,pop_2024_censo,2024,ratio_pct
0,Pando,Manuripi,Filadelfia,7993.0,8671.0,8.482422
1,Pando,Federico Román,Villa Nueva,2485.0,2730.0,9.859155
2,Pando,Abuná,Ingavi,2545.0,2800.0,10.019646
3,Pando,Abuná,Santa Rosa,2827.0,3123.0,10.470463
4,Pando,Madre de Dios,San Lorenzo,9602.0,10622.0,10.622787
5,Pando,Federico Román,Nueva Esperanza,1572.0,1743.0,10.877863
6,Pando,Madre de Dios,Sena,10247.0,11371.0,10.969064
7,Pando,Federico Román,Santos Mercado,2282.0,2536.0,11.130587
8,Pando,Nicolás Suárez,Porvenir,9096.0,10121.0,11.268690
9,Beni,General José Ballivián,Santa Rosa,10953.0,12221.0,11.576737


### 8.2 Backtest de rho et du logistique sur l'agrégat régional

In [45]:
taux_region_0112 = 100 * math.log(pop_region[2012] / pop_region[2001]) / 11

ref_obs = rate_nat.loc[[2010, 2011, 2012]].mean()
YEARS_BT = list(range(2013, 2025))
rho_obs = {y: rate_nat.loc[y] / ref_obs for y in YEARS_BT}

prev = pop_region[2012]
for y in YEARS_BT:
    prev = prev * (1 + taux_region_0112 * rho_obs[y] / 100)
pred_2024_rho_agg = prev
ecart_rho_agg = 100 * (pred_2024_rho_agg - pop_region[2024]) / pop_region[2024]
print(f"rho agrégé : prédiction 2024 = {pred_2024_rho_agg:.0f}  vs observé {pop_region[2024]:.0f}  écart = {ecart_rho_agg:.3f} %")

K_agg = 2 * pop_region[2012]
ab_agg = solve_ab(K_agg, pop_region[2001], pop_region[2012], 11)
pred_2024_log_agg = logistic(K_agg, ab_agg[0], ab_agg[1], 2024, 2001)
ecart_log_agg = 100 * (pred_2024_log_agg - pop_region[2024]) / pop_region[2024]
print(f"logistique agrégé (K=2·P2012) : prédiction 2024 = {pred_2024_log_agg:.0f}  écart = {ecart_log_agg:.3f} %")

rho agrégé : prédiction 2024 = 337586  vs observé 317517  écart = 6.321 %
logistique agrégé (K=2·P2012) : prédiction 2024 = 347625  écart = 9.482 %


### 8.3 Architecture combinée (implémentée, pas substituée à la projection principale)

**a. Total régional** — ratio INE agrégé `I_région(t) = ΣINE_i(t) / ΣINE_i(2024)` converti sur
base Censo (`pop_région[2024]·I_région(t)`), taux à 2034 prolongé par `rho` (profil national,
étape 1).

In [46]:
I_region = {t: m[str(t)].sum() / m["2024"].sum() for t in range(2024, 2035)}
region_total_a = {t: pop_region[2024] * I_region[t] for t in range(2024, 2035)}
assert abs(region_total_a[2024] - pop_region[2024]) < 1e-6

taux_region_2034 = 100 * math.log(region_total_a[2034] / region_total_a[2033])
prev = region_total_a[2034]
for y in YEARS_FUT:
    prev = prev * (1 + taux_region_2034 * rho[y] / 100)
    region_total_a[y] = prev
print(f"taux régional (ratio INE agrégé) à 2034 = {taux_region_2034:.4f} %/an")
print(f"total régional a : 2035 = {region_total_a[2035]:.0f}   2050 = {region_total_a[2050]:.0f}")

taux régional (ratio INE agrégé) à 2034 = 0.8215 %/an
total régional a : 2035 = 349496   2050 = 381128


**b. Distribution municipale** — logistique de parts calée sur les parts INE 2024/2034
(`part_i(t) = INE_i(t)/ΣINE(t)`), même jeu d'asymptotes `[2.0, 0.5]` qu'aux étapes 6-7. Contrôle
= total régional du point a, **pas** de total national. Renormalisation à chaque année cible pour
que les parts somment à 1 (assertion).

In [47]:
part_2024 = m["2024"] / m["2024"].sum()
part_2034 = m["2034"] / m["2034"].sum()

raw_parts = {2035: [], 2050: []}
factors_used = []
for i in range(len(m)):
    p0, p1 = part_2024.iloc[i], part_2034.iloc[i]
    K_p, f_p, ab = solve_with_factors(p0, p1, 10, K_FACTORS, p1)
    factors_used.append(f_p)
    for t in (2035, 2050):
        raw_parts[t].append(logistic(K_p, ab[0], ab[1], t, 2024))

print("facteurs utilisés :", pd.Series(factors_used).value_counts().to_dict())

parts_final = {}
for t in (2035, 2050):
    s = pd.Series(raw_parts[t])
    parts_final[t] = s / s.sum()
    assert abs(parts_final[t].sum() - 1) < 1e-9, f"STOP somme parts {t}"
print("somme des parts après renormalisation : 1.0 (assertion OK) pour 2035 et 2050")

facteurs utilisés : {2.0: 14, 0.5: 7}
somme des parts après renormalisation : 1.0 (assertion OK) pour 2035 et 2050


**c. Population municipale = part × total régional.** Continuité à 2034 : `part_i(2034)` (part
INE exacte) × `total_régional_a(2034)` comparé à `pop[2034]` de l'étape 1 (base censo ×
`I_i(2034)` propre à chaque municipalité). Ce n'est **pas** une assertion bloquante ici — si
l'écart dépasse 0,1 %, c'est affiché explicitement plutôt que masqué, parce qu'un `assert` qui
lève interromprait l'exécution avant le tableau demandé au point 3.

In [48]:
pop_a_2035 = parts_final[2035] * region_total_a[2035]
pop_a_2050 = parts_final[2050] * region_total_a[2050]

pop_a_2034_check = part_2034.values * region_total_a[2034]
rel_err_2034 = np.abs(pop_a_2034_check - pop[2034].values) / pop[2034].values
print(f"écart max continuité 2034 = {rel_err_2034.max()*100:.2f} %  (seuil 0,1 %)")
if rel_err_2034.max() > 0.001:
    print("ÉCHEC : la continuité à 0,1 % près N'EST PAS satisfaite.")
    print("Cause : le rapport INE/Censo varie par municipalité (8,48 % à 17,18 %, point 8.1),")
    print("donc le total région 'a' (ratio agrégé) et la somme des pop[2034] de l'étape 1")
    print("(ratios individuels) ne coïncident pas exactement.")

pd.DataFrame({
    "municipio": m["MUNICIPIO/TIOC"],
    "pop_a_2034_check": pop_a_2034_check,
    "pop_etape1_2034": pop[2034].values,
    "rel_err_pct": rel_err_2034 * 100,
}).sort_values("rel_err_pct", ascending=False)

écart max continuité 2034 = 4.11 %  (seuil 0,1 %)
ÉCHEC : la continuité à 0,1 % près N'EST PAS satisfaite.
Cause : le rapport INE/Censo varie par municipalité (8,48 % à 17,18 %, point 8.1),
donc le total région 'a' (ratio agrégé) et la somme des pop[2034] de l'étape 1
(ratios individuels) ne coïncident pas exactement.


,municipio,pop_a_2034_check,pop_etape1_2034,rel_err_pct
12,Filadelfia,8670.724535,9042.017876,4.106311
11,San Pedro,3783.331737,3652.506569,3.581791
19,Villa Nueva,2376.073764,2446.769231,2.889339
17,Ingavi,2800.372650,2879.485714,2.747472
10,Puerto Rico,7815.939070,7625.475772,2.497723
16,Santa Rosa,3054.951982,3128.438040,2.348970
14,San Lorenzo,10594.212820,10834.115044,2.214322
2,Guayaramerín,42712.754566,41847.370352,2.067954
9,Bella Flor,3417.373948,3349.945795,2.012813
18,Nueva Esperanza,1495.653575,1526.003442,1.988847


Tableau par cluster, 2035 et 2050 — taux annuels implicites et part régionale dans le national.

In [49]:
combo = pd.DataFrame({
    "municipio": m["MUNICIPIO/TIOC"].values,
    "cluster_label": base_cl["cluster_label"].values,
    "pop_2024": m["pop_2024_censo"].values,
    "pop_a_2035": pop_a_2035.values,
    "pop_a_2050": pop_a_2050.values,
})
cluster_combo = combo.groupby("cluster_label")[["pop_2024", "pop_a_2035", "pop_a_2050"]].sum()
cluster_combo["taux_2024_2035"] = 100 * np.log(cluster_combo["pop_a_2035"] / cluster_combo["pop_2024"]) / 11
cluster_combo["taux_2035_2050"] = 100 * np.log(cluster_combo["pop_a_2050"] / cluster_combo["pop_a_2035"]) / 15
cluster_combo.round(2)

,pop_2024,pop_a_2035,pop_a_2050,taux_2024_2035,taux_2035_2050
cluster_label,,,,,
C1,41377.0,43559.62,45477.25,0.47,0.29
C2,2338.0,2576.21,2806.98,0.88,0.57
C3,159706.0,175243.92,189710.17,0.84,0.53
C4,62188.0,68461.97,76051.65,0.87,0.70
C5,51908.0,59653.81,67081.81,1.26,0.78


In [50]:
nat_2024 = float(pop_nat.loc[2024])
part_nation = pd.Series({
    2024: pop_region[2024] / nat_2024,
    2035: region_total_a[2035] / float(pop_nat.loc[2035]),
    2050: region_total_a[2050] / float(pop_nat.loc[2050]),
}, name="part_région_dans_nation")
part_nation

2024    0.026645
2035    0.028709
2050    0.030860
Name: part_région_dans_nation, dtype: float64

### 8.4

Projection principale des étapes 1-5 non substituée — architecture combinée implémentée à titre
de comparaison uniquement, en attente de décision.

## Étape 9 — architecture combinée corrigée (8.3 était sur-déterminée)

### 9.1 2024-2034 : étape 1 réutilisée telle quelle

Aucun total top-down. `pop[2024..2034]` (étape 1) reste la seule source : chaque municipalité
part de son Censo 2024 et suit son propre `I_i(t)`, le total régional est la somme.

### 9.2 Extension au-delà de 2034

**a. Total régional** — taux agrégé implicite de `Σpop[t]` (étape 1) entre 2033 et 2034,
prolongé par `rho`.

In [51]:
region_total = {t: pop[t].sum() for t in range(2024, 2035)}
assert abs(region_total[2024] - pop_region[2024]) < 1e-6

taux_region_2034 = 100 * math.log(region_total[2034] / region_total[2033])
prev = region_total[2034]
for y in YEARS_FUT:
    prev = prev * (1 + taux_region_2034 * rho[y] / 100)
    region_total[y] = prev
print(f"taux régional (agrégat Σpop étape 1) à 2034 = {taux_region_2034:.4f} %/an")
print(f"total régional : 2035 = {region_total[2035]:.0f}   2050 = {region_total[2050]:.0f}")

taux régional (agrégat Σpop étape 1) à 2034 = 0.8213 %/an
total régional : 2035 = 349534   2050 = 381165


**b. Parts municipales** — logistique calée non sur le niveau des parts INE mais sur leur
**trajectoire relative** `R_i(t) = S_i(t)/S_i(2034)` (`R_i(2034) = 1` par construction), même jeu
d'asymptotes `[2.0, 0.5]`. Cette trajectoire relative est appliquée à la part réelle 2034 issue de
l'étape 1 (`pop[2034]_i / Σpop[2034]`, cohérente Censo), puis renormalisée à 1.

In [52]:
S_2024 = m["2024"] / m["2024"].sum()
S_2034 = m["2034"] / m["2034"].sum()
R_2024 = S_2024 / S_2034  # R_i(2024) ; R_i(2034) = 1 trivialement

raw_R = {2035: [], 2050: []}
factors_used = []
for i in range(len(m)):
    K_r, f_r, ab = solve_with_factors(R_2024.iloc[i], 1.0, 10, K_FACTORS, 1.0)
    factors_used.append(f_r)
    for t in (2035, 2050):
        raw_R[t].append(logistic(K_r, ab[0], ab[1], t, 2024))
print("facteurs utilisés :", pd.Series(factors_used).value_counts().to_dict())

part_2034_real = pop[2034] / pop[2034].sum()

parts_final = {}
for t in (2035, 2050):
    raw_part_t = part_2034_real.values * np.array(raw_R[t])
    parts_final[t] = raw_part_t / raw_part_t.sum()
    assert abs(parts_final[t].sum() - 1) < 1e-9, f"STOP somme parts {t}"  # assertion bloquante
print("parts renormalisées : somme = 1 (assertion OK) pour 2035 et 2050")

facteurs utilisés : {2.0: 14, 0.5: 7}
parts renormalisées : somme = 1 (assertion OK) pour 2035 et 2050


**c. Population municipale = part × total régional (a).**

In [53]:
pop_9_2035 = parts_final[2035] * region_total[2035]
pop_9_2050 = parts_final[2050] * region_total[2050]

# continuité 2034 : part réelle (R=1) x total régional 2034, doit retomber exactement sur pop[2034]
pop_9_2034_check = part_2034_real.values * region_total[2034]
rel_err_2034 = np.abs(pop_9_2034_check - pop[2034].values) / pop[2034].values
print(f"écart max continuité 2034 = {rel_err_2034.max()*100:.6f} %")
assert rel_err_2034.max() < 0.0001, f"STOP continuité : {rel_err_2034.max()}"  # < 0.01 %, bloquante

assert (pop_9_2035 > 0).all() and (pop_9_2050 > 0).all(), "STOP population négative"  # bloquante

print("trois assertions OK : continuité < 0,01 %, parts = 1 à 1e-9 près, aucune population négative")

écart max continuité 2034 = 0.000000 %
trois assertions OK : continuité < 0,01 %, parts = 1 à 1e-9 près, aucune population négative


### 9.3 Le tableau qui décide — rho (étapes 1-5) vs architecture combinée, par cluster

In [54]:
combo9 = pd.DataFrame({
    "municipio": m["MUNICIPIO/TIOC"].values,
    "cluster_label": base_cl["cluster_label"].values,
    "rho_2035": pop[2035].values, "rho_2050": pop[2050].values,
    "combo_2035": pop_9_2035, "combo_2050": pop_9_2050,
})
cluster_tab9 = combo9.groupby("cluster_label")[["rho_2035", "combo_2035", "rho_2050", "combo_2050"]].sum()
cluster_tab9["ecart_2035_pct"] = 100 * (cluster_tab9["combo_2035"] - cluster_tab9["rho_2035"]) / cluster_tab9["rho_2035"]
cluster_tab9["ecart_2050_pct"] = 100 * (cluster_tab9["combo_2050"] - cluster_tab9["rho_2050"]) / cluster_tab9["rho_2050"]

region_row9 = pd.DataFrame({c: [cluster_tab9[c].sum()] for c in ["rho_2035", "combo_2035", "rho_2050", "combo_2050"]}, index=["RÉGION"])
region_row9["ecart_2035_pct"] = 100 * (region_row9["combo_2035"] - region_row9["rho_2035"]) / region_row9["rho_2035"]
region_row9["ecart_2050_pct"] = 100 * (region_row9["combo_2050"] - region_row9["rho_2050"]) / region_row9["rho_2050"]

decision_table = pd.concat([cluster_tab9, region_row9]).round(2)
decision_table

,rho_2035,combo_2035,rho_2050,combo_2050,ecart_2035_pct,ecart_2050_pct
C1,43743.01,43706.68,46445.02,45620.72,-0.08,-1.77
C2,2587.98,2580.99,2909.95,2811.70,-0.27,-3.38
C3,175343.83,175173.47,192393.83,189663.96,-0.10,-1.42
C4,69245.25,69379.38,74970.42,77077.28,0.19,2.81
C5,58615.10,58693.91,64557.44,65991.01,0.13,2.22
RÉGION,349535.19,349534.43,381276.66,381164.67,-0.00,-0.03


### 9.4 Décision

In [55]:
max_ecart = cluster_tab9[["ecart_2035_pct", "ecart_2050_pct"]].abs().values.max()
print(f"écart maximal par cluster (hors ligne RÉGION) = {max_ecart:.3f} %")

if max_ecart < 3:
    pop_main = pop.copy()
    pop_main[2035] = pop_9_2035
    pop_main[2050] = pop_9_2050
    print("SUBSTITUTION : écart < 3 % — l'architecture combinée remplace la projection principale (2035, 2050).")
else:
    over = cluster_tab9[cluster_tab9[["ecart_2035_pct", "ecart_2050_pct"]].abs().max(axis=1) >= 3]
    print(f"PAS DE SUBSTITUTION : écart ≥ 3 % pour {list(over.index)} — projection principale (rho, étapes 1-5) conservée telle quelle.")

écart maximal par cluster (hors ligne RÉGION) = 3.376 %
PAS DE SUBSTITUTION : écart ≥ 3 % pour ['C2'] — projection principale (rho, étapes 1-5) conservée telle quelle.


## Étape 10 — substitution et ménages

**Substitution** : l'architecture combinée de l'étape 9 remplace la projection principale. Le
seuil de 3 % non pondéré de l'étape 9 ne se déclenchait que sur C2 (98 habitants, Bolpebra
seule) ; critère retenu désormais : le backtest par cluster.

In [56]:
pop_final_2035 = pop_9_2035
pop_final_2050 = pop_9_2050
print("projection principale = architecture combinée (étape 9)")
print(f"région 2035 = {pop_final_2035.sum():.0f}   région 2050 = {pop_final_2050.sum():.0f}")

projection principale = architecture combinée (étape 9)
région 2035 = 349534   région 2050 = 381165


### 10.1 Écart de décohabitation régional (vérification, pas hypothèse)

`ecart(période) = taux_ménages − taux_population`, log-annualisé, `CSV_final` uniquement.

In [57]:
HH_BLOCK = "NÚMERO DE VIVIENDAS POR FUENTE DE ELECTRICIDAD"
HH_SUBCOL_BY_YEAR = {2001: "Electricidad total", 2012: "Total", 2024: "Total"}

for year, subcol in HH_SUBCOL_BY_YEAR.items():
    col = f"{HH_BLOCK} | {year} | {subcol}"
    base[f"hh_{year}_censo"] = municipios[col].astype(float).values

hh_region = {year: float(amazonia[f"{HH_BLOCK} | {year} | {subcol}"].iloc[0]) for year, subcol in HH_SUBCOL_BY_YEAR.items()}

taux_pop_0112 = 100 * math.log(pop_region[2012] / pop_region[2001]) / 11
taux_hh_0112 = 100 * math.log(hh_region[2012] / hh_region[2001]) / 11
ecart_0112_reg = taux_hh_0112 - taux_pop_0112

taux_pop_1224 = 100 * math.log(pop_region[2024] / pop_region[2012]) / 12
taux_hh_1224 = 100 * math.log(hh_region[2024] / hh_region[2012]) / 12
ecart_1224_reg = taux_hh_1224 - taux_pop_1224

facteur_reg = ecart_1224_reg / ecart_0112_reg

print(f"ecart 2001-2012 = {ecart_0112_reg:.3f} pts/an  (attendu 1,687)")
print(f"ecart 2012-2024 = {ecart_1224_reg:.3f} pts/an  (attendu 1,083)")
print(f"facteur         = {facteur_reg:.3f}            (attendu 0,642)")

assert abs(ecart_0112_reg - 1.687) < 0.01 and abs(ecart_1224_reg - 1.083) < 0.01 and abs(facteur_reg - 0.642) < 0.01, \
    "STOP : les chiffres régionaux ne correspondent pas à l'attendu"
print("vérifié.")

ecart 2001-2012 = 1.687 pts/an  (attendu 1,687)
ecart 2012-2024 = 1.083 pts/an  (attendu 1,083)
facteur         = 0.642            (attendu 0,642)
vérifié.


### 10.2 Deux groupes — écart et facteur propres

`urbain` = Cobija, Riberalta, Guayaramerín ; `reste` = les 18 autres.

In [58]:
URBAIN = ["Cobija", "Riberalta", "Guayaramerín"]
base["groupe"] = np.where(base["MUNICIPIO/TIOC"].isin(URBAIN), "urbain", "reste")

grp = base.groupby("groupe")[["pop_2001_censo", "pop_2012_censo", "pop_2024_censo",
                               "hh_2001_censo", "hh_2012_censo", "hh_2024_censo"]].sum()

group_params = {}
for g in ["urbain", "reste"]:
    r = grp.loc[g]
    e0112 = 100 * math.log(r["hh_2012_censo"] / r["hh_2001_censo"]) / 11 - 100 * math.log(r["pop_2012_censo"] / r["pop_2001_censo"]) / 11
    e1224 = 100 * math.log(r["hh_2024_censo"] / r["hh_2012_censo"]) / 12 - 100 * math.log(r["pop_2024_censo"] / r["pop_2012_censo"]) / 12
    group_params[g] = dict(ecart_1224=e1224, facteur=e1224 / e0112)
    print(f"{g:8s}: ecart_0112={e0112:.4f}  ecart_1224={e1224:.4f}  facteur={e1224/e0112:.4f}")

urbain  : ecart_0112=1.8726  ecart_1224=0.9344  facteur=0.4990
reste   : ecart_0112=1.3993  ecart_1224=1.3727  facteur=0.9810


### 10.3 Prolongement

`ecart(t) = ecart(2012-2024)·facteur^((t-2024)/12)` par groupe. `taux_population_i(t)` = taux
moyen 2024→2035 (11 ans) pour t≤2035, taux moyen 2035→2050 (15 ans) pour t>2035, issus de la
projection principale substituée. Chaînage annuel depuis `hh_2024_censo`.

In [59]:
taux_pop_2435 = 100 * np.log(pop_final_2035 / base["pop_2024_censo"].values) / 11
taux_pop_3550 = 100 * np.log(pop_final_2050 / pop_final_2035) / 15
groupe_arr = base["groupe"].values
hh0 = base["hh_2024_censo"].values

hh = hh0.astype(float).copy()
hh_main_2035 = None
for t in range(2025, 2051):
    taux_pop_t = np.where(t <= 2035, taux_pop_2435, taux_pop_3550)
    ecart_t = np.array([group_params[g]["ecart_1224"] * group_params[g]["facteur"] ** ((t - 2024) / 12) for g in groupe_arr])
    hh = hh * (1 + (taux_pop_t + ecart_t) / 100)
    if t == 2035:
        hh_main_2035 = hh.copy()
hh_main_2050 = hh

print(f"ménages région : 2024 = {hh0.sum():.0f}   2035 = {hh_main_2035.sum():.0f}   2050 = {hh_main_2050.sum():.0f}")

ménages région : 2024 = 84209   2035 = 102236   2050 = 123710


### 10.4a Taille de ménage implicite

In [60]:
taille_2024 = base["pop_2024_censo"].values / hh0
taille_2035 = pop_final_2035 / hh_main_2035
taille_2050 = pop_final_2050 / hh_main_2050

taille_table = pd.DataFrame({
    "municipio": base["MUNICIPIO/TIOC"], "groupe": base["groupe"],
    "taille_2024": taille_2024, "taille_2035": taille_2035, "taille_2050": taille_2050,
}).sort_values("taille_2050")

taille_region_2024 = base["pop_2024_censo"].sum() / hh0.sum()
taille_region_2035 = pop_final_2035.sum() / hh_main_2035.sum()
taille_region_2050 = pop_final_2050.sum() / hh_main_2050.sum()
print(f"taille régionale : 2024 = {taille_region_2024:.3f}   2035 = {taille_region_2035:.3f}   2050 = {taille_region_2050:.3f}  (attendu ~3,3)")
taille_table

taille régionale : 2024 = 3.771   2035 = 3.419   2050 = 3.081  (attendu ~3,3)


,municipio,groupe,taille_2024,taille_2035,taille_2050
9,Bella Flor,reste,2.770040,2.386965,1.956306
8,Bolpebra,reste,2.915212,2.517193,2.066929
12,Filadelfia,reste,3.179395,2.747622,2.258248
18,Nueva Esperanza,reste,3.214724,2.769867,2.269951
3,Reyes,reste,3.302312,2.844826,2.331076
16,Santa Rosa,reste,3.287209,2.839140,2.331977
0,Ixiamas,reste,3.427102,2.958336,2.428371
10,Puerto Rico,reste,3.453997,2.983423,2.450696
19,Villa Nueva,reste,3.509887,3.024705,2.479111
15,Sena,reste,3.564174,3.079986,2.531273


### 10.4b Assertion — aucune taille sous 2,5 ni au-dessus de 2024

In [61]:
viol = taille_table[(taille_table["taille_2035"] < 2.5) | (taille_table["taille_2050"] < 2.5) |
                     (taille_table["taille_2035"] > taille_table["taille_2024"]) | (taille_table["taille_2050"] > taille_table["taille_2024"])]

if len(viol):
    print(f"ÉCHEC : {len(viol)} municipalité(s) violent l'assertion (toutes 'reste', plancher 2,5) :")
    print(viol[["municipio", "groupe", "taille_2024", "taille_2035", "taille_2050"]].to_string())
    print("Cause : facteur 'reste' ≈ 0,98 — l'écart de décohabitation ne se résorbe presque pas sur 26 ans,")
    print("il reste ≈ 1,32 pt/an en 2050 (contre 1,37 en 2024) et finit par pousser plusieurs petites")
    print("municipalités rurales sous le plancher.")
else:
    print("assertion OK")

ÉCHEC : 9 municipalité(s) violent l'assertion (toutes 'reste', plancher 2,5) :
          municipio groupe  taille_2024  taille_2035  taille_2050
9        Bella Flor  reste     2.770040     2.386965     1.956306
8          Bolpebra  reste     2.915212     2.517193     2.066929
12       Filadelfia  reste     3.179395     2.747622     2.258248
18  Nueva Esperanza  reste     3.214724     2.769867     2.269951
3             Reyes  reste     3.302312     2.844826     2.331076
16       Santa Rosa  reste     3.287209     2.839140     2.331977
0           Ixiamas  reste     3.427102     2.958336     2.428371
10      Puerto Rico  reste     3.453997     2.983423     2.450696
19      Villa Nueva  reste     3.509887     3.024705     2.479111
Cause : facteur 'reste' ≈ 0,98 — l'écart de décohabitation ne se résorbe presque pas sur 26 ans,
il reste ≈ 1,32 pt/an en 2050 (contre 1,37 en 2024) et finit par pousser plusieurs petites
municipalités rurales sous le plancher.


### 10.4c Contrôle croisé — extension directe du taux de ménages 2012-2024 via rho

Même mécanique que la population : taux constant jusqu'en 2034, `rho(t)` (national, inchangé)
à partir de 2035.

In [62]:
taux_hh_1224_muni = 100 * np.log(base["hh_2024_censo"].values / base["hh_2012_censo"].values) / 12

hh_direct = base["hh_2024_censo"].values.astype(float).copy()
hh_direct_2035 = None
for t in range(2025, 2051):
    taux_t = taux_hh_1224_muni if t <= 2034 else taux_hh_1224_muni * rho[t]
    hh_direct = hh_direct * (1 + taux_t / 100)
    if t == 2035:
        hh_direct_2035 = hh_direct.copy()
hh_direct_2050 = hh_direct

combo_hh = pd.DataFrame({
    "cluster_label": base_cl["cluster_label"].values,
    "hh_main_2035": hh_main_2035, "hh_main_2050": hh_main_2050,
    "hh_direct_2035": hh_direct_2035, "hh_direct_2050": hh_direct_2050,
})
cluster_hh = combo_hh.groupby("cluster_label").sum()
cluster_hh["ecart_2035_pct"] = 100 * (cluster_hh["hh_direct_2035"] - cluster_hh["hh_main_2035"]) / cluster_hh["hh_main_2035"]
cluster_hh["ecart_2050_pct"] = 100 * (cluster_hh["hh_direct_2050"] - cluster_hh["hh_main_2050"]) / cluster_hh["hh_main_2050"]

over5 = cluster_hh[(cluster_hh["ecart_2035_pct"].abs() > 5) | (cluster_hh["ecart_2050_pct"].abs() > 5)]
print("clusters > 5 % d'écart :", list(over5.index) if len(over5) else "aucun")
cluster_hh.round(2)

clusters > 5 % d'écart : ['C1', 'C2', 'C3', 'C4', 'C5']


,hh_main_2035,hh_main_2050,hh_direct_2035,hh_direct_2050,ecart_2035_pct,ecart_2050_pct
cluster_label,,,,,,
C1,13300.46,16810.36,14404.74,18972.81,8.30,12.86
C2,1025.35,1360.33,1135.44,1589.28,10.74,16.83
C3,47622.36,54454.03,53316.00,70588.43,11.96,29.63
C4,21373.86,28792.27,23641.57,33887.49,10.61,17.70
C5,18914.10,22293.13,19255.32,23651.18,1.80,6.09


### 10.4d Tableau final — ménages par cluster

In [63]:
cluster_hh_final = pd.DataFrame({
    "cluster_label": base_cl["cluster_label"].values,
    "hh_2024": hh0, "hh_2035": hh_main_2035, "hh_2050": hh_main_2050,
}).groupby("cluster_label").sum()
cluster_hh_final.round(0)

,hh_2024,hh_2035,hh_2050
cluster_label,,,
C1,10933.0,13300.0,16810.0
C2,802.0,1025.0,1360.0
C3,40298.0,47622.0,54454.0
C4,16612.0,21374.0,28792.0
C5,15564.0,18914.0,22293.0


### 10.5 Sensibilité — facteur de décroissance unique donnant une taille régionale 2050 cible

Un seul facteur (remplace les deux facteurs de groupe) résolu par `brentq` pour atteindre 2,9
puis 3,6.

In [64]:
def hh_regional_size_2050(factor_override):
    gp = {g: dict(ecart_1224=group_params[g]["ecart_1224"], facteur=factor_override) for g in ("urbain", "reste")}
    hh_s = hh0.astype(float).copy()
    for t in range(2025, 2051):
        taux_pop_t = np.where(t <= 2035, taux_pop_2435, taux_pop_3550)
        ecart_t = np.array([gp[g]["ecart_1224"] * gp[g]["facteur"] ** ((t - 2024) / 12) for g in groupe_arr])
        hh_s = hh_s * (1 + (taux_pop_t + ecart_t) / 100)
    return pop_final_2050.sum() / hh_s.sum()

def hh_cluster_table(factor_override):
    gp = {g: dict(ecart_1224=group_params[g]["ecart_1224"], facteur=factor_override) for g in ("urbain", "reste")}
    hh_s = hh0.astype(float).copy()
    hh_s_2035 = None
    for t in range(2025, 2051):
        taux_pop_t = np.where(t <= 2035, taux_pop_2435, taux_pop_3550)
        ecart_t = np.array([gp[g]["ecart_1224"] * gp[g]["facteur"] ** ((t - 2024) / 12) for g in groupe_arr])
        hh_s = hh_s * (1 + (taux_pop_t + ecart_t) / 100)
        if t == 2035:
            hh_s_2035 = hh_s.copy()
    df = pd.DataFrame({"cluster_label": base_cl["cluster_label"].values, "hh_2035": hh_s_2035, "hh_2050": hh_s})
    return df.groupby("cluster_label").sum()

f_29 = brentq(lambda f: hh_regional_size_2050(f) - 2.9, 0.5, 1.4)
f_36 = brentq(lambda f: hh_regional_size_2050(f) - 3.6, -0.5, 0.3)
print(f"facteur -> taille régionale 2050 = 2,9 : {f_29:.4f}")
print(f"facteur -> taille régionale 2050 = 3,6 : {f_36:.4f}")

sens = pd.concat({
    "naturel (0,499 / 0,981)": cluster_hh_final[["hh_2035", "hh_2050"]],
    "facteur unique -> 2,9": hh_cluster_table(f_29),
    "facteur unique -> 3,6": hh_cluster_table(f_36),
}, axis=1).round(0)
sens

facteur -> taille régionale 2050 = 2,9 : 0.9535
facteur -> taille régionale 2050 = 3,6 : 0.0998


c:\Users\valen\anaconda3\Lib\site-packages\scipy\optimize\_zeros_py.py:846: ComplexWarning: Casting complex values to real discards the imaginary part
  r = _zeros._brentq(f, a, b, xtol, rtol, maxiter, args, full_output, disp)


naturel (0,499 / 0,981)          facteur unique -> 2,9           \
                              hh_2035  hh_2050               hh_2035  hh_2050   
cluster_label                                                                   
C1                            13300.0  16810.0               13273.0  16631.0   
C2                             1025.0   1360.0                1023.0   1346.0   
C3                            47622.0  54454.0               48809.0  60165.0   
C4                            21374.0  28792.0               21330.0  28487.0   
C5                            18914.0  22293.0               19413.0  24807.0   

              facteur unique -> 3,6           
                            hh_2035  hh_2050  
cluster_label                                 
C1                          12139.0  12681.0  
C2                            936.0   1027.0  
C3                          45866.0  49785.0  
C4                          19517.0  21747.0  
C5                          18273.0  20638.0

## Étape 11 — facteur unique régional (correctif de l'étape 10.2)

Le facteur par groupe (10.2) est estimé sur 2 points : 0,981 pour « reste » signifie une
décohabitation qui ne décélère quasiment pas en 26 ans — un artefact d'estimation, pas un
signal démographique. Correctif : un seul facteur, le régional 0,642 (étape 10.1, calé sur
l'agrégat, le plus robuste), appliqué aux deux groupes. Seul l'écart de départ reste
différencié : 0,934 pts/an (urbain), 1,373 (reste).

In [65]:
group_params_11 = {g: dict(ecart_1224=group_params[g]["ecart_1224"], facteur=facteur_reg) for g in ("urbain", "reste")}
print(group_params_11)

hh11 = hh0.astype(float).copy()
hh11_2035 = None
for t in range(2025, 2051):
    taux_pop_t = np.where(t <= 2035, taux_pop_2435, taux_pop_3550)
    ecart_t = np.array([group_params_11[g]["ecart_1224"] * group_params_11[g]["facteur"] ** ((t - 2024) / 12) for g in groupe_arr])
    hh11 = hh11 * (1 + (taux_pop_t + ecart_t) / 100)
    if t == 2035:
        hh11_2035 = hh11.copy()
hh11_2050 = hh11
print(f"ménages région (corrigé) : 2024 = {hh0.sum():.0f}   2035 = {hh11_2035.sum():.0f}   2050 = {hh11_2050.sum():.0f}")

{'urbain': {'ecart_1224': 0.9343984151719154, 'facteur': 0.642015971451374}, 'reste': {'ecart_1224': 1.372728319851811, 'facteur': 0.642015971451374}}
ménages région (corrigé) : 2024 = 84209   2035 = 101783   2050 = 120199


### 1. Taille de ménage régionale et par municipalité

In [66]:
taille11_2024 = base["pop_2024_censo"].values / hh0
taille11_2035 = pop_final_2035 / hh11_2035
taille11_2050 = pop_final_2050 / hh11_2050

taille11_table = pd.DataFrame({
    "municipio": base["MUNICIPIO/TIOC"], "groupe": base["groupe"],
    "taille_2024": taille11_2024, "taille_2035": taille11_2035, "taille_2050": taille11_2050,
}).sort_values("taille_2050")

print(f"taille régionale (corrigée) : 2024 = {base['pop_2024_censo'].sum()/hh0.sum():.3f}   "
      f"2035 = {pop_final_2035.sum()/hh11_2035.sum():.3f}   2050 = {pop_final_2050.sum()/hh11_2050.sum():.3f}  (attendu ~3,14)")
taille11_table

taille régionale (corrigée) : 2024 = 3.771   2035 = 3.434   2050 = 3.171  (attendu ~3,14)


,municipio,groupe,taille_2024,taille_2035,taille_2050
9,Bella Flor,reste,2.770040,2.453451,2.212739
8,Bolpebra,reste,2.915212,2.586535,2.335254
12,Filadelfia,reste,3.179395,2.823070,2.550456
18,Nueva Esperanza,reste,3.214724,2.847085,2.567681
16,Santa Rosa,reste,3.287209,2.917270,2.634384
3,Reyes,reste,3.302312,2.924267,2.637180
0,Ixiamas,reste,3.427102,3.039930,2.744016
10,Puerto Rico,reste,3.453997,3.065499,2.768402
19,Villa Nueva,reste,3.509887,3.108909,2.803945
15,Sena,reste,3.564174,3.164576,2.858871


### 2. Assertion plancher 2,5 — combien de municipalités la violent encore ?

In [67]:
viol11 = taille11_table[(taille11_table["taille_2035"] < 2.5) | (taille11_table["taille_2050"] < 2.5) |
                         (taille11_table["taille_2035"] > taille11_table["taille_2024"]) | (taille11_table["taille_2050"] > taille11_table["taille_2024"])]
print(f"{len(viol11)} municipalité(s) violent encore l'assertion (contre 9 à l'étape 10) :")
viol11[["municipio", "groupe", "taille_2024", "taille_2035", "taille_2050"]]

2 municipalité(s) violent encore l'assertion (contre 9 à l'étape 10) :


,municipio,groupe,taille_2024,taille_2035,taille_2050
9,Bella Flor,reste,2.770040,2.453451,2.212739
8,Bolpebra,reste,2.915212,2.586535,2.335254


### 3. Contrôle croisé — écart par cluster (affiché sans correction)

In [68]:
combo11 = pd.DataFrame({
    "cluster_label": base_cl["cluster_label"].values,
    "hh_main_2035": hh11_2035, "hh_main_2050": hh11_2050,
    "hh_direct_2035": hh_direct_2035, "hh_direct_2050": hh_direct_2050,
})
cluster_hh11 = combo11.groupby("cluster_label").sum()
cluster_hh11["ecart_2035_pct"] = 100 * (cluster_hh11["hh_direct_2035"] - cluster_hh11["hh_main_2035"]) / cluster_hh11["hh_main_2035"]
cluster_hh11["ecart_2050_pct"] = 100 * (cluster_hh11["hh_direct_2050"] - cluster_hh11["hh_main_2050"]) / cluster_hh11["hh_main_2050"]

over5_11 = cluster_hh11[(cluster_hh11["ecart_2035_pct"].abs() > 5) | (cluster_hh11["ecart_2050_pct"].abs() > 5)]
print("clusters > 5 % (non corrigé, documenté) :", list(over5_11.index) if len(over5_11) else "aucun")
cluster_hh11.round(2)

clusters > 5 % (non corrigé, documenté) : ['C1', 'C2', 'C3', 'C4']


,hh_main_2035,hh_main_2050,hh_direct_2035,hh_direct_2050,ecart_2035_pct,ecart_2050_pct
cluster_label,,,,,,
C1,12942.35,14872.78,14404.74,18972.81,11.30,27.57
C2,997.86,1204.02,1135.44,1589.28,13.79,32.00
C3,47957.76,55641.01,53316.00,70588.43,11.17,26.86
C4,20801.24,25487.30,23641.57,33887.49,13.65,32.96
C5,19083.76,22993.88,19255.32,23651.18,0.90,2.86


### 4. Tableau ménages par cluster, méthode corrigée

In [69]:
cluster_hh11_final = pd.DataFrame({
    "cluster_label": base_cl["cluster_label"].values,
    "hh_2024": hh0, "hh_2035": hh11_2035, "hh_2050": hh11_2050,
}).groupby("cluster_label").sum()
cluster_hh11_final.round(0)

,hh_2024,hh_2035,hh_2050
cluster_label,,,
C1,10933.0,12942.0,14873.0
C2,802.0,998.0,1204.0
C3,40298.0,47958.0,55641.0
C4,16612.0,20801.0,25487.0
C5,15564.0,19084.0,22994.0


### 5. Sensibilité 2,9 / 3,6 — facteur unique remplaçant 0,642

In [70]:
def hh_regional_size_2050_11(factor_override):
    gp = {g: dict(ecart_1224=group_params[g]["ecart_1224"], facteur=factor_override) for g in ("urbain", "reste")}
    hh_s = hh0.astype(float).copy()
    for t in range(2025, 2051):
        taux_pop_t = np.where(t <= 2035, taux_pop_2435, taux_pop_3550)
        ecart_t = np.array([gp[g]["ecart_1224"] * gp[g]["facteur"] ** ((t - 2024) / 12) for g in groupe_arr])
        hh_s = hh_s * (1 + (taux_pop_t + ecart_t) / 100)
    return pop_final_2050.sum() / hh_s.sum()

def hh_cluster_table_11(factor_override):
    gp = {g: dict(ecart_1224=group_params[g]["ecart_1224"], facteur=factor_override) for g in ("urbain", "reste")}
    hh_s = hh0.astype(float).copy()
    hh_s_2035 = None
    for t in range(2025, 2051):
        taux_pop_t = np.where(t <= 2035, taux_pop_2435, taux_pop_3550)
        ecart_t = np.array([gp[g]["ecart_1224"] * gp[g]["facteur"] ** ((t - 2024) / 12) for g in groupe_arr])
        hh_s = hh_s * (1 + (taux_pop_t + ecart_t) / 100)
        if t == 2035:
            hh_s_2035 = hh_s.copy()
    df = pd.DataFrame({"cluster_label": base_cl["cluster_label"].values, "hh_2035": hh_s_2035, "hh_2050": hh_s})
    return df.groupby("cluster_label").sum()

f11_29 = brentq(lambda f: hh_regional_size_2050_11(f) - 2.9, 0.6, 1.3)
f11_36 = brentq(lambda f: hh_regional_size_2050_11(f) - 3.6, 0.0, 0.55)
print(f"facteur -> taille régionale 2050 = 2,9 : {f11_29:.4f}")
print(f"facteur -> taille régionale 2050 = 3,6 : {f11_36:.4f}")

sens11 = pd.concat({
    "corrigé (0,642)": cluster_hh11_final[["hh_2035", "hh_2050"]],
    "facteur unique -> 2,9": hh_cluster_table_11(f11_29),
    "facteur unique -> 3,6": hh_cluster_table_11(f11_36),
}, axis=1).round(0)
sens11

facteur -> taille régionale 2050 = 2,9 : 0.9535
facteur -> taille régionale 2050 = 3,6 : 0.0998


corrigé (0,642)          facteur unique -> 2,9           \
                      hh_2035  hh_2050               hh_2035  hh_2050   
cluster_label                                                           
C1                    12942.0  14873.0               13273.0  16631.0   
C2                      998.0   1204.0                1023.0   1346.0   
C3                    47958.0  55641.0               48809.0  60165.0   
C4                    20801.0  25487.0               21330.0  28487.0   
C5                    19084.0  22994.0               19413.0  24807.0   

              facteur unique -> 3,6           
                            hh_2035  hh_2050  
cluster_label                                 
C1                          12139.0  12681.0  
C2                            936.0   1027.0  
C3                          45866.0  49785.0  
C4                          19517.0  21747.0  
C5                          18273.0  20638.0

### 6. Livrable — `projections/output/menages_projetes.csv`

Méthode corrigée (facteur régional unique 0,642).

In [71]:
import os

livrable = pd.DataFrame({
    "municipio": base["MUNICIPIO/TIOC"].values,
    "departamento": base["DEPARTAMENTO"].values,
    "provincia": base["PROVINCIA"].values,
    "cluster": base_cl["cluster_label"].values,
    "pop_2024": base["pop_2024_censo"].values,
    "pop_2035": pop_final_2035,
    "pop_2050": pop_final_2050,
    "hh_2024": hh0,
    "hh_2035": hh11_2035,
    "hh_2050": hh11_2050,
})
livrable["hhsize_2024"] = livrable["pop_2024"] / livrable["hh_2024"]
livrable["hhsize_2035"] = livrable["pop_2035"] / livrable["hh_2035"]
livrable["hhsize_2050"] = livrable["pop_2050"] / livrable["hh_2050"]

os.makedirs("output", exist_ok=True)
livrable.to_csv("output/menages_projetes.csv", index=False)
print(f"écrit : output/menages_projetes.csv  ({len(livrable)} lignes)")
livrable

écrit : output/menages_projetes.csv  (21 lignes)


,municipio,departamento,provincia,cluster,pop_2024,pop_2035,pop_2050,hh_2024,hh_2035,hh_2050,hhsize_2024,hhsize_2035,hhsize_2050
0,Ixiamas,La Paz,Abel Iturralde,C1,11330.0,12340.082859,13202.987533,3306.0,4059.330684,4811.556447,3.427102,3.039930,2.744016
1,Riberalta,Beni,Vaca Diez,C3,107816.0,119705.385078,131414.159852,27442.0,33049.840302,38893.263225,3.928868,3.621966,3.378841
2,Guayaramerín,Beni,Vaca Diez,C3,40130.0,41989.276423,42834.477434,10891.0,12371.561766,13537.038630,3.684694,3.394016,3.164243
3,Reyes,Beni,General José Ballivián,C1,11284.0,10725.881687,10036.290708,3417.0,3667.887598,3805.690712,3.302312,2.924267,2.637180
4,Santa Rosa,Beni,General José Ballivián,C1,10953.0,11979.229072,12887.062621,2755.0,3396.620710,4047.827522,3.975681,3.526808,3.183699
5,Exaltación,Beni,Yacuma,C1,7810.0,8661.484503,9494.380890,1455.0,1818.512577,2207.703755,5.367698,4.762950,4.300568
6,Cobija,Pando,Nicolás Suárez,C5,51908.0,58693.910848,65991.009181,15564.0,19083.756416,22993.877001,3.335132,3.075595,2.869938
7,Porvenir,Pando,Nicolás Suárez,C4,9096.0,10259.294623,11497.637099,2230.0,2833.554181,3515.947217,4.078924,3.620645,3.270139
8,Bolpebra,Pando,Nicolás Suárez,C2,2338.0,2580.991446,2811.701379,802.0,997.856669,1204.023590,2.915212,2.586535,2.335254
9,Bella Flor,Pando,Nicolás Suárez,C4,3421.0,3343.390205,3197.224132,1235.0,1362.729332,1444.916876,2.770040,2.453451,2.212739
